In [357]:
from pathlib import Path

pages_path = Path("streamlit_app/pages")

In [1]:
from pathlib import Path

page_path = Path("streamlit_app/pages/1_Team_Tournament.py")

team_tournament_code = r'''
import ast
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import streamlit as st

from utils.data_loader import load_all_data
from utils.filters import get_tournament_options, get_team_options, filter_team
from utils.style import apply_global_style
from utils.cards import metric_card, text_card, list_card, app_footer

from utils.visuals import (
    plot_team_xi_pitch,
    plot_team_shot_map,
    plot_shot_zones,
    plot_passing_zones,
    plot_carry_zones,
    plot_recovery_zones,
    plot_pressure_zones,
    plot_foul_zones,
    plot_team_similarity_bar,
    plot_team_radar_comparison,
    plot_team_dna_bars,
    plot_box_entry_zones_by_type,
    plot_cross_origin_zones,
    plot_cross_target_zones,
    plot_offensive_duel_zones,
    plot_defensive_duel_zones,
    plot_set_piece_shot_zones,
    plot_goal_timing,
    plot_long_pass_origin_zones,
    plot_long_pass_target_zones,
    plot_progressive_pass_zones,
    plot_progressive_carry_zones,
    plot_conceded_shot_zones,
    plot_aerial_duel_zones,
    plot_set_piece_profile
)


st.set_page_config(page_title="Team Tournament", layout="wide")
apply_global_style()

data = load_all_data()

matches = data["matches"]
events = data["events"]
team_master = data["team_master"]
player_master = data["player_master"]
most_frequent_xi = data["most_frequent_xi"]
team_similarity = data["team_similarity"]
assets = data["assets"]

if "event_type" not in events.columns and "type" in events.columns:
    events["event_type"] = events["type"]

BASE_DIR = Path(__file__).resolve().parents[2]


# ============================================================
# HELPERS
# ============================================================

position_order = {
    "Goalkeeper": 1,
    "Right Back": 2,
    "Right Center Back": 3,
    "Center Back": 4,
    "Left Center Back": 5,
    "Left Back": 6,
    "Right Wing Back": 7,
    "Left Wing Back": 8,
    "Right Defensive Midfield": 9,
    "Center Defensive Midfield": 10,
    "Left Defensive Midfield": 11,
    "Right Center Midfield": 12,
    "Center Midfield": 13,
    "Left Center Midfield": 14,
    "Right Midfield": 15,
    "Left Midfield": 16,
    "Right Attacking Midfield": 17,
    "Center Attacking Midfield": 18,
    "Left Attacking Midfield": 19,
    "Right Wing": 20,
    "Left Wing": 21,
    "Right Center Forward": 22,
    "Center Forward": 23,
    "Left Center Forward": 24,
    "Secondary Striker": 25
}

line_mapping = {
    "Goalkeeper": "Goalkeeper",
    "Right Back": "Defenders",
    "Right Center Back": "Defenders",
    "Center Back": "Defenders",
    "Left Center Back": "Defenders",
    "Left Back": "Defenders",
    "Right Wing Back": "Defenders",
    "Left Wing Back": "Defenders",
    "Right Defensive Midfield": "Midfielders",
    "Center Defensive Midfield": "Midfielders",
    "Left Defensive Midfield": "Midfielders",
    "Right Center Midfield": "Midfielders",
    "Center Midfield": "Midfielders",
    "Left Center Midfield": "Midfielders",
    "Right Midfield": "Midfielders",
    "Left Midfield": "Midfielders",
    "Right Attacking Midfield": "Midfielders",
    "Center Attacking Midfield": "Midfielders",
    "Left Attacking Midfield": "Midfielders",
    "Right Wing": "Forwards",
    "Left Wing": "Forwards",
    "Right Center Forward": "Forwards",
    "Center Forward": "Forwards",
    "Left Center Forward": "Forwards",
    "Secondary Striker": "Forwards"
}

line_order = {
    "Goalkeeper": 1,
    "Defenders": 2,
    "Midfielders": 3,
    "Forwards": 4,
    "Other": 99
}


def get_team_asset(team_name):
    asset_row = assets[assets["team"] == team_name]

    if len(asset_row) == 0:
        return None, "#2563EB"

    primary_color = asset_row.iloc[0].get("primary_color", "#2563EB")
    raw_flag_path = asset_row.iloc[0].get("flag_path", None)

    if not raw_flag_path or pd.isna(raw_flag_path):
        return None, primary_color

    raw_flag_path = str(raw_flag_path)

    possible_paths = [
        BASE_DIR / raw_flag_path,
        BASE_DIR / "streamlit_app" / raw_flag_path,
        Path(raw_flag_path)
    ]

    for p in possible_paths:
        if p.exists():
            return str(p), primary_color

    return None, primary_color


def prepare_xi_table(most_frequent_xi, tournament_id, team):
    xi = most_frequent_xi[
        (most_frequent_xi["tournament_id"] == tournament_id) &
        (most_frequent_xi["team"] == team)
    ].copy()

    if len(xi) == 0:
        return xi

    xi["position_order"] = xi["position"].map(position_order).fillna(99)

    sort_cols = ["starts"]
    ascending_values = [False]

    if "importance_score" in xi.columns:
        sort_cols.append("importance_score")
        ascending_values.append(False)

    xi = xi.sort_values(sort_cols, ascending=ascending_values).copy()

    selected_rows = []
    used_positions = set()
    used_players = set()

    for _, row in xi.iterrows():
        if row["position"] not in used_positions and row["player"] not in used_players:
            selected_rows.append(row)
            used_positions.add(row["position"])
            used_players.add(row["player"])

        if len(selected_rows) == 11:
            break

    if len(selected_rows) < 11:
        for _, row in xi.iterrows():
            if row["player"] not in used_players:
                selected_rows.append(row)
                used_players.add(row["player"])

            if len(selected_rows) == 11:
                break

    xi = pd.DataFrame(selected_rows)

    if len(xi) == 0:
        return xi

    xi["position_order"] = xi["position"].map(position_order).fillna(99)
    xi = xi.sort_values("position_order").head(11).copy()
    xi["#"] = range(1, len(xi) + 1)

    return xi


def prepare_squad_table(player_master, tournament_id, team):
    squad = player_master[
        (player_master["tournament_id"] == tournament_id) &
        (player_master["team"] == team)
    ].copy()

    if len(squad) == 0:
        return squad

    squad["line"] = squad["position"].map(line_mapping).fillna("Other")
    squad["line_order"] = squad["line"].map(line_order).fillna(99)
    squad["position_order"] = squad["position"].map(position_order).fillna(99)

    sort_cols = ["line_order", "position_order"]
    ascending_values = [True, True]

    if "importance_score" in squad.columns:
        sort_cols.append("importance_score")
        ascending_values.append(False)

    return squad.sort_values(sort_cols, ascending=ascending_values)


def render_top_player_cards(df, value_col, subtitle_col="general_role", top_n=5):
    if len(df) == 0 or value_col not in df.columns:
        return

    players = df.sort_values(value_col, ascending=False).head(top_n)
    cols = st.columns(top_n)

    for i, (_, p) in enumerate(players.iterrows()):
        with cols[i]:
            metric_card(
                p["player"],
                round(p.get(value_col, 0), 1),
                p.get(subtitle_col, "")
            )


def generate_team_summary(row, team, tournament_id):
    scores = {
        "attacking threat": row.get("attack_score", 0),
        "build-up control": row.get("build_up_score", 0),
        "defensive activity": row.get("defense_score", 0),
        "directness": row.get("directness_score", 0)
    }

    main_strength = max(scores, key=scores.get)

    return (
        f"{team} shows a {row.get('general_style', 'N/A')} profile in {tournament_id}. "
        f"Offensively, the team is classified as {row.get('offensive_style', 'N/A')}, "
        f"while defensively it profiles as {row.get('defensive_style', 'N/A')}. "
        f"The strongest team DNA indicator is {main_strength}."
    )


def count_box_entries(events, tournament_id, team):
    df = events[
        (events["tournament_id"] == tournament_id) &
        (events["team"] == team) &
        (events["type"].isin(["Pass", "Carry"]))
    ].copy()

    if len(df) == 0:
        return 0, 0, 0

    def parse_loc(v):
        if isinstance(v, list):
            return v
        if isinstance(v, str):
            try:
                return ast.literal_eval(v)
            except Exception:
                return None
        return None

    loc = df["location"].apply(parse_loc)
    df["x"] = loc.apply(lambda v: v[0] if isinstance(v, list) and len(v) > 0 else None)
    df["y"] = loc.apply(lambda v: v[1] if isinstance(v, list) and len(v) > 1 else None)

    df["end_x"] = None
    df["end_y"] = None

    if "pass_end_location" in df.columns:
        pass_end = df["pass_end_location"].apply(parse_loc)
        df["pass_end_x"] = pass_end.apply(lambda v: v[0] if isinstance(v, list) and len(v) > 0 else None)
        df["pass_end_y"] = pass_end.apply(lambda v: v[1] if isinstance(v, list) and len(v) > 1 else None)
        df["end_x"] = df["end_x"].fillna(df["pass_end_x"])
        df["end_y"] = df["end_y"].fillna(df["pass_end_y"])

    if "carry_end_location" in df.columns:
        carry_end = df["carry_end_location"].apply(parse_loc)
        df["carry_end_x"] = carry_end.apply(lambda v: v[0] if isinstance(v, list) and len(v) > 0 else None)
        df["carry_end_y"] = carry_end.apply(lambda v: v[1] if isinstance(v, list) and len(v) > 1 else None)
        df["end_x"] = df["end_x"].fillna(df["carry_end_x"])
        df["end_y"] = df["end_y"].fillna(df["carry_end_y"])

    entries = df[
        (df["end_x"] >= 102) &
        (df["end_y"] >= 18) &
        (df["end_y"] <= 62) &
        ~(
            (df["x"] >= 102) &
            (df["y"] >= 18) &
            (df["y"] <= 62)
        )
    ].copy()

    pass_entries = len(entries[entries["type"] == "Pass"])
    carry_entries = len(entries[entries["type"] == "Carry"])

    crosses = 0
    if "pass_cross" in entries.columns:
        crosses = len(entries[entries["pass_cross"].fillna(False) == True])

    return pass_entries, carry_entries, crosses


def get_team_match_ids(matches, tournament_id, team):
    team_matches = matches[
        (matches["tournament_id"] == tournament_id) &
        (
            (matches["home_team"] == team) |
            (matches["away_team"] == team)
        )
    ].copy()

    return team_matches["match_id"].dropna().unique()


def plot_goal_timing_team_tournament(events, matches, tournament_id, team, team_color="#2563EB"):
    match_ids = get_team_match_ids(matches, tournament_id, team)

    shots = events[
        (events["match_id"].isin(match_ids)) &
        (events["type"] == "Shot") &
        (events["shot_outcome"] == "Goal")
    ].copy()

    if len(shots) == 0 or "minute" not in shots.columns:
        return None

    shots["goal_type"] = shots["team"].apply(
        lambda x: "Goals For" if x == team else "Goals Against"
    )

    bins = [0, 15, 30, 45, 60, 75, 90, 120]
    labels = ["0-15", "16-30", "31-45", "46-60", "61-75", "76-90", "90+"]

    shots["period_bin"] = pd.cut(
        shots["minute"],
        bins=bins,
        labels=labels,
        include_lowest=True
    )

    pivot = (
        shots
        .groupby(["period_bin", "goal_type"], observed=False)
        .size()
        .unstack(fill_value=0)
        .reindex(labels, fill_value=0)
    )

    for col in ["Goals For", "Goals Against"]:
        if col not in pivot.columns:
            pivot[col] = 0

    fig, ax = plt.subplots(figsize=(8, 4))

    x = np.arange(len(labels))
    width = 0.36

    ax.bar(
        x - width / 2,
        pivot["Goals For"],
        width,
        label="Goals For",
        color=team_color,
        alpha=0.9
    )

    ax.bar(
        x + width / 2,
        pivot["Goals Against"],
        width,
        label="Goals Against",
        color="#DC2626",
        alpha=0.9
    )

    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45)
    ax.set_title(f"{team} | Goal Timing", fontsize=15, weight="bold")
    ax.set_xlabel("Match Period")
    ax.set_ylabel("Goals")
    ax.legend(loc="upper left")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    return fig


def plot_team_style_profile(events, matches, tournament_id, team, team_color="#2563EB"):
    match_ids = get_team_match_ids(matches, tournament_id, team)

    df = events[
        (events["match_id"].isin(match_ids)) &
        (events["team"] == team)
    ].copy()

    if len(df) == 0:
        return None

    crosses = 0
    if "pass_cross" in df.columns:
        crosses = len(
            df[
                (df["type"] == "Pass") &
                (df["pass_cross"].fillna(False) == True)
            ]
        )

    progressive_passes = 0
    if "pass_length" in df.columns:
        progressive_passes = len(
            df[
                (df["type"] == "Pass") &
                (df["pass_length"].fillna(0) >= 30)
            ]
        )

    values = {
        "Shots": len(df[df["type"] == "Shot"]),
        "Passes": len(df[df["type"] == "Pass"]),
        "Carries": len(df[df["type"] == "Carry"]),
        "Crosses": crosses,
        "Progressive Passes": progressive_passes,
        "Pressures": len(df[df["type"] == "Pressure"]),
        "Recoveries": len(df[df["type"].isin(["Ball Recovery", "Interception"])])
    }

    plot_df = pd.DataFrame({
        "metric": list(values.keys()),
        "value": list(values.values())
    }).sort_values("value", ascending=True)

    fig, ax = plt.subplots(figsize=(8, 4))

    ax.barh(
        plot_df["metric"],
        plot_df["value"],
        color=team_color,
        alpha=0.85
    )

    ax.set_title(f"{team} | Team Style Profile", fontsize=15, weight="bold")
    ax.set_xlabel("Events")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    return fig


def get_team_transition_data(events, tournament_id, team):
    df = events[events["tournament_id"] == tournament_id].copy()

    if "index" not in df.columns:
        df = df.reset_index().rename(columns={"index": "event_index"})
        idx_col = "event_index"
    else:
        idx_col = "index"

    rows_passes_before_shot = []
    rows_recovery_to_shot = []

    for match_id, match_df in df.groupby("match_id"):
        match_df = match_df.sort_values(idx_col)

        shots = match_df[
            (match_df["team"] == team) &
            (match_df["type"] == "Shot")
        ].copy()

        for _, shot in shots.iterrows():
            previous = match_df[
                (match_df[idx_col] < shot[idx_col]) &
                (match_df[idx_col] >= shot[idx_col] - 25)
            ]

            passes = len(
                previous[
                    (previous["team"] == team) &
                    (previous["type"] == "Pass")
                ]
            )

            rows_passes_before_shot.append(passes)

        recoveries = match_df[
            (match_df["team"] == team) &
            (match_df["type"].isin(["Ball Recovery", "Interception"]))
        ].copy()

        for _, recovery in recoveries.iterrows():
            next_shots = shots[shots[idx_col] > recovery[idx_col]]

            if len(next_shots) == 0:
                continue

            next_shot = next_shots.iloc[0]

            if "minute" not in recovery.index or "minute" not in next_shot.index:
                continue

            seconds = (next_shot["minute"] - recovery["minute"]) * 60

            if 0 <= seconds <= 30:
                rows_recovery_to_shot.append(seconds)

    return rows_passes_before_shot, rows_recovery_to_shot


def plot_passes_before_shot_buckets(passes_before_shot, team, team_color="#2563EB"):
    if len(passes_before_shot) == 0:
        return None

    bins = [0, 2, 4, 6, 8, 100]
    labels = ["0-2", "3-4", "5-6", "7-8", "9+"]

    df = pd.DataFrame({"passes": passes_before_shot})
    df["bucket"] = pd.cut(df["passes"], bins=bins, labels=labels, include_lowest=True, right=True)

    plot_df = df["bucket"].value_counts().reindex(labels, fill_value=0).reset_index()
    plot_df.columns = ["bucket", "shots"]

    fig, ax = plt.subplots(figsize=(8, 4))

    ax.bar(plot_df["bucket"], plot_df["shots"], color=team_color, alpha=0.85)

    ax.set_title(f"{team} | Passes Before Shot", fontsize=15, weight="bold")
    ax.set_xlabel("Passes before shot")
    ax.set_ylabel("Shots")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    return fig


def plot_recovery_to_shot_buckets(recovery_to_shot, team, team_color="#2563EB"):
    if len(recovery_to_shot) == 0:
        return None

    bins = [0, 5, 10, 15, 20, 30]
    labels = ["0-5s", "5-10s", "10-15s", "15-20s", "20-30s"]

    df = pd.DataFrame({"seconds": recovery_to_shot})
    df["bucket"] = pd.cut(df["seconds"], bins=bins, labels=labels, include_lowest=True, right=True)

    plot_df = df["bucket"].value_counts().reindex(labels, fill_value=0).reset_index()
    plot_df.columns = ["bucket", "sequences"]

    plot_df["percentage"] = (plot_df["sequences"] / plot_df["sequences"].sum() * 100).fillna(0)

    fig, ax = plt.subplots(figsize=(8, 4))

    ax.barh(
        plot_df["bucket"][::-1],
        plot_df["percentage"][::-1],
        color=team_color,
        alpha=0.85
    )

    ax.set_title(f"{team} | Recovery to Shot Speed", fontsize=15, weight="bold")
    ax.set_xlabel("% of sequences")

    for i, value in enumerate(plot_df["percentage"][::-1]):
        ax.text(value + 1, i, f"{value:.1f}%", va="center", fontsize=9, weight="bold")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    return fig


def plot_similarity_bar_safe(team_similarity, tournament_id, team, primary_color):
    try:
        return plot_team_similarity_bar(team_similarity, tournament_id, team, team_color=primary_color)
    except TypeError:
        return plot_team_similarity_bar(team_similarity, tournament_id, team)


# ============================================================
# SIDEBAR
# ============================================================

st.sidebar.title("Filters")

tournament_id = st.sidebar.selectbox("Tournament", get_tournament_options(matches))
team = st.sidebar.selectbox("Team", get_team_options(team_master, tournament_id))

team_row = filter_team(team_master, tournament_id, team)
flag_path, primary_color = get_team_asset(team)


# ============================================================
# HEADER
# ============================================================

col_flag, col_title, col_empty = st.columns([1, 6, 1])

with col_flag:
    if flag_path:
        st.image(flag_path, width=90)

with col_title:
    st.markdown(
        f"<h1 style='color:{primary_color}; margin-bottom:0; text-align:center;'>{team}</h1>",
        unsafe_allow_html=True
    )
    st.markdown(
        f"<div class='page-subtitle' style='text-align:center;'>{tournament_id} · Team Tournament Analysis</div>",
        unsafe_allow_html=True
    )


# ============================================================
# KPI ROW
# ============================================================

if len(team_row) > 0:
    row = team_row.iloc[0]

    c1, c2, c3, c4, c5 = st.columns(5)

    with c1:
        metric_card("Matches", int(row.get("matches", 0)))

    with c2:
        metric_card("Wins", int(row.get("wins", 0)))

    with c3:
        metric_card("Losses", int(row.get("losses", 0)))

    with c4:
        metric_card("Goals For", int(row.get("goals_for", row.get("goals", 0))))

    with c5:
        metric_card("Goals Against", int(row.get("goals_against", 0)))

    c6, c7, c8, c9 = st.columns(4)

    with c6:
        metric_card("Attack Score", round(row.get("attack_score", 0), 1))

    with c7:
        metric_card("Build-Up", round(row.get("build_up_score", 0), 1))

    with c8:
        metric_card("Defense", round(row.get("defense_score", 0), 1))

    with c9:
        metric_card("Directness", round(row.get("directness_score", 0), 1))


# ============================================================
# TABS
# ============================================================

tab_overview, tab_build, tab_attack, tab_defense, tab_transitions, tab_setpieces, tab_players, tab_similarity = st.tabs([
    "Overview",
    "Build-Up",
    "Attack",
    "Defense",
    "Transitions",
    "Set Pieces",
    "Players",
    "Similarity"
])


# ============================================================
# OVERVIEW
# ============================================================

with tab_overview:

    st.markdown("### Team Structure")

    pitch_col, xi_col = st.columns([1.05, 1])

    with pitch_col:
        fig = plot_team_xi_pitch(
            most_frequent_xi,
            tournament_id,
            team,
            team_color=primary_color
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)

    with xi_col:
        xi = prepare_xi_table(most_frequent_xi, tournament_id, team)

        if len(xi) > 0:
            xi_cols = [
                "#",
                "player",
                "position",
                "general_role",
                "offensive_role",
                "defensive_role",
                "starts",
                "importance_score",
                "importance_tier"
            ]

            existing_xi_cols = [c for c in xi_cols if c in xi.columns]

            st.dataframe(
                xi[existing_xi_cols],
                use_container_width=True,
                hide_index=True,
                height=420
            )
        else:
            st.info("No XI table available.")

    st.markdown("### Tactical Summary & Team DNA")

    summary_col, dna_col = st.columns([1, 1])

    with summary_col:

        if len(team_row) > 0:
            row = team_row.iloc[0]

            text_card(
                "Auto Scouting Summary",
                generate_team_summary(row, team, tournament_id)
            )

            text_card(
                "General Style",
                row.get("general_style", "N/A")
            )

            text_card(
                "Offensive Style",
                row.get("offensive_style", "N/A")
            )

            text_card(
                "Defensive Style",
                row.get("defensive_style", "N/A")
            )

            if "strengths" in row.index:
                list_card(
                    "Team Strengths",
                    row.get("strengths", [])
                )

            if "weaknesses" in row.index:
                list_card(
                    "Team Weaknesses",
                    row.get("weaknesses", [])
                )

    with dna_col:
        fig = plot_team_dna_bars(
            team_master,
            tournament_id,
            team,
            team_color=primary_color
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)

    st.markdown("### Match Dynamics")

    md1, md2 = st.columns(2)

    with md1:
        fig = plot_goal_timing_team_tournament(
            events,
            matches,
            tournament_id,
            team,
            team_color=primary_color
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No goal timing available.")

    with md2:
        fig = plot_team_style_profile(
            events,
            matches,
            tournament_id,
            team,
            team_color=primary_color
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No team style profile available.")

    st.markdown("### Key Players")

    key_players = player_master[
        (player_master["tournament_id"] == tournament_id) &
        (player_master["team"] == team)
    ].copy()

    render_top_player_cards(key_players, "importance_score")


# ============================================================
# BUILD-UP
# ============================================================

with tab_build:

    st.markdown("### Build-Up & Progression")

    if len(team_row) > 0:
        row = team_row.iloc[0]

        c1, c2, c3, c4 = st.columns(4)

        with c1:
            metric_card("Passes / Match", round(row.get("passes_per_match", 0), 2))

        with c2:
            metric_card("Pass Accuracy", round(row.get("avg_pass_completion_pct", 0), 1))

        with c3:
            metric_card("Tempo", round(row.get("tempo", 0), 2))

        with c4:
            metric_card("Build-Up Score", round(row.get("build_up_score", 0), 1))

    b1, b2 = st.columns(2)

    with b1:
        fig = plot_passing_zones(events, tournament_id, team)
        if fig is not None:
            st.pyplot(fig, use_container_width=True)

    with b2:
        fig = plot_carry_zones(events, tournament_id, team)
        if fig is not None:
            st.pyplot(fig, use_container_width=True)

    st.markdown("### Long Passing")

    lp1, lp2 = st.columns(2)

    with lp1:
        fig = plot_long_pass_origin_zones(events, tournament_id, team)
        if fig is not None:
            st.pyplot(fig, use_container_width=True)

    with lp2:
        fig = plot_long_pass_target_zones(events, tournament_id, team)
        if fig is not None:
            st.pyplot(fig, use_container_width=True)

    st.markdown("### Progression")

    pr1, pr2 = st.columns(2)

    with pr1:
        fig = plot_progressive_pass_zones(events, tournament_id, team)
        if fig is not None:
            st.pyplot(fig, use_container_width=True)

    with pr2:
        fig = plot_progressive_carry_zones(events, tournament_id, team)
        if fig is not None:
            st.pyplot(fig, use_container_width=True)

    st.markdown("### Best Build-Up Players")

    control_players = player_master[
        (player_master["tournament_id"] == tournament_id) &
        (player_master["team"] == team)
    ].copy()

    render_top_player_cards(control_players, "passing_volume")


# ============================================================
# ATTACK
# ============================================================

with tab_attack:

    st.markdown("### Attacking Identity")

    pass_entries, carry_entries, cross_entries = count_box_entries(
        events,
        tournament_id,
        team
    )

    a1, a2, a3, a4 = st.columns(4)

    with a1:
        metric_card("Passes Into Box", pass_entries)

    with a2:
        metric_card("Carries Into Box", carry_entries)

    with a3:
        metric_card("Crosses Into Box", cross_entries)

    with a4:
        metric_card("Total Box Entries", pass_entries + carry_entries)

    st.markdown("### Shooting")

    r1c1, r1c2 = st.columns(2)

    with r1c1:
        fig = plot_team_shot_map(
            events,
            tournament_id,
            team,
            team_color=primary_color
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)

    with r1c2:
        fig = plot_shot_zones(
            events,
            tournament_id,
            team
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)

    st.markdown("### Crossing")

    r2c1, r2c2 = st.columns(2)

    with r2c1:
        fig = plot_cross_origin_zones(
            events,
            tournament_id,
            team
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)

    with r2c2:
        fig = plot_cross_target_zones(
            events,
            tournament_id,
            team
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)

    st.markdown("### Penalty Area Access")

    r3c1, r3c2 = st.columns(2)

    with r3c1:
        fig = plot_box_entry_zones_by_type(
            events,
            tournament_id,
            team,
            team_color=primary_color
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)

    with r3c2:
        fig = plot_offensive_duel_zones(
            events,
            tournament_id,
            team
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)

    st.markdown("### Most Dangerous Players")

    attacking_players = player_master[
        (player_master["tournament_id"] == tournament_id) &
        (player_master["team"] == team)
    ]

    render_top_player_cards(attacking_players, "attacking_volume")


# ============================================================
# DEFENSE
# ============================================================

with tab_defense:

    st.markdown("### Defensive Structure")

    if len(team_row) > 0:
        row = team_row.iloc[0]

        d1, d2, d3, d4 = st.columns(4)

        with d1:
            metric_card("Pressures / Match", round(row.get("pressures_per_match", 0), 2))

        with d2:
            metric_card("Recoveries / Match", round(row.get("recoveries_per_match", 0), 2))

        with d3:
            metric_card("Defensive Activity", round(row.get("defensive_activity", 0), 2))

        with d4:
            metric_card("Defense Score", round(row.get("defense_score", 0), 1))

    st.markdown("### Pressing & Recoveries")

    r1c1, r1c2 = st.columns(2)

    with r1c1:
        fig = plot_recovery_zones(events, tournament_id, team)
        if fig is not None:
            st.pyplot(fig, use_container_width=True)

    with r1c2:
        fig = plot_pressure_zones(events, tournament_id, team)
        if fig is not None:
            st.pyplot(fig, use_container_width=True)

    st.markdown("### Defensive Duels & Fouls")

    r2c1, r2c2 = st.columns(2)

    with r2c1:
        fig = plot_defensive_duel_zones(events, tournament_id, team)
        if fig is not None:
            st.pyplot(fig, use_container_width=True)

    with r2c2:
        fig = plot_foul_zones(events, tournament_id, team)
        if fig is not None:
            st.pyplot(fig, use_container_width=True)

    st.markdown("### Defensive Vulnerability")

    r3c1, r3c2 = st.columns(2)

    with r3c1:
        fig = plot_conceded_shot_zones(events, tournament_id, team)
        if fig is not None:
            st.pyplot(fig, use_container_width=True)

    with r3c2:
        fig = plot_aerial_duel_zones(events, tournament_id, team)
        if fig is not None:
            st.pyplot(fig, use_container_width=True)

    st.markdown("### Defensive Leaders")

    defensive_players = player_master[
        (player_master["tournament_id"] == tournament_id) &
        (player_master["team"] == team)
    ]

    render_top_player_cards(defensive_players, "defensive_volume")


# ============================================================
# TRANSITIONS
# ============================================================

with tab_transitions:

    st.markdown("### Transition Profile")

    passes_before_shot, recovery_to_shot = get_team_transition_data(
        events,
        tournament_id,
        team
    )

    avg_passes = round(np.mean(passes_before_shot), 2) if len(passes_before_shot) > 0 else 0
    median_passes = round(np.median(passes_before_shot), 2) if len(passes_before_shot) > 0 else 0
    avg_recovery_speed = round(np.mean(recovery_to_shot), 2) if len(recovery_to_shot) > 0 else 0

    fast_attacks_pct = 0

    if len(recovery_to_shot) > 0:
        fast_attacks_pct = round(
            len([x for x in recovery_to_shot if x <= 10]) / len(recovery_to_shot) * 100,
            1
        )

    k1, k2, k3, k4 = st.columns(4)

    with k1:
        metric_card("Avg Passes Before Shot", avg_passes)

    with k2:
        metric_card("Median Passes", median_passes)

    with k3:
        metric_card("Avg Recovery → Shot", f"{avg_recovery_speed}s")

    with k4:
        metric_card("Fast Attacks ≤10s", f"{fast_attacks_pct}%")

    text_card(
        "Transition Reading",
        "This section evaluates how directly the team attacks before shooting and how quickly it creates shots after recoveries."
    )

    tr1, tr2 = st.columns(2)

    with tr1:
        fig = plot_passes_before_shot_buckets(
            passes_before_shot,
            team,
            team_color=primary_color
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No passes-before-shot data available.")

    with tr2:
        fig = plot_recovery_to_shot_buckets(
            recovery_to_shot,
            team,
            team_color=primary_color
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No recovery-to-shot data available.")


# ============================================================
# SET PIECES
# ============================================================

with tab_setpieces:

    st.markdown("### Set Pieces")

    team_events = events[
        (events["tournament_id"] == tournament_id) &
        (events["team"] == team)
    ].copy()

    if "play_pattern" in team_events.columns:
        corners = len(team_events[team_events["play_pattern"] == "From Corner"])
        free_kicks = len(team_events[team_events["play_pattern"] == "From Free Kick"])
        throw_ins = len(team_events[team_events["play_pattern"] == "From Throw In"])
    else:
        corners = 0
        free_kicks = 0
        throw_ins = 0

    s1, s2, s3 = st.columns(3)

    with s1:
        metric_card("Corner Events", corners)

    with s2:
        metric_card("Free Kick Events", free_kicks)

    with s3:
        metric_card("Throw-In Events", throw_ins)

    sp1, sp2 = st.columns(2)

    with sp1:
        fig = plot_set_piece_shot_zones(events, tournament_id, team)

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No set piece shot zones available.")

    with sp2:
        fig = plot_set_piece_profile(events, tournament_id, team)

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No set piece profile available.")

    st.markdown("### Main Set Piece Takers")

    taker_col1, taker_col2 = st.columns(2)

    with taker_col1:
        st.markdown("#### Corner Takers")

        if "play_pattern" in team_events.columns and "player" in team_events.columns:
            corner_df = team_events[team_events["play_pattern"] == "From Corner"].copy()

            if len(corner_df) > 0:
                corner_takers = (
                    corner_df
                    .dropna(subset=["player"])
                    .groupby("player")
                    .size()
                    .reset_index(name="corners")
                    .sort_values("corners", ascending=False)
                    .head(10)
                )

                st.dataframe(corner_takers, use_container_width=True, hide_index=True)
            else:
                st.info("No corner takers available.")
        else:
            st.info("Corner taker data not available.")

    with taker_col2:
        st.markdown("#### Free Kick Takers")

        if "play_pattern" in team_events.columns and "player" in team_events.columns:
            free_kick_df = team_events[team_events["play_pattern"] == "From Free Kick"].copy()

            if len(free_kick_df) > 0:
                free_kick_takers = (
                    free_kick_df
                    .dropna(subset=["player"])
                    .groupby("player")
                    .size()
                    .reset_index(name="free_kicks")
                    .sort_values("free_kicks", ascending=False)
                    .head(10)
                )

                st.dataframe(free_kick_takers, use_container_width=True, hide_index=True)
            else:
                st.info("No free kick takers available.")
        else:
            st.info("Free kick taker data not available.")

    st.markdown("### Main Set Piece Targets")

    if "play_pattern" in team_events.columns and "pass_recipient" in team_events.columns:
        target_df = team_events[
            team_events["play_pattern"].isin([
                "From Corner",
                "From Free Kick",
                "From Throw In"
            ])
        ].copy()

        target_df = target_df.dropna(subset=["pass_recipient"])

        if len(target_df) > 0:
            targets = (
                target_df
                .groupby("pass_recipient")
                .size()
                .reset_index(name="times_targeted")
                .sort_values("times_targeted", ascending=False)
                .head(10)
            )

            st.dataframe(targets, use_container_width=True, hide_index=True)
        else:
            st.info("No set piece target data available.")
    else:
        st.info("Set piece target data not available.")


# ============================================================
# PLAYERS
# ============================================================

with tab_players:

    st.markdown("### Player Roles")

    squad_table = prepare_squad_table(player_master, tournament_id, team)

    st.markdown("#### Most Important Players")
    render_top_player_cards(squad_table, "importance_score")

    st.markdown("#### Most Dangerous Players")
    render_top_player_cards(squad_table, "attacking_volume")

    st.markdown("#### Best Build-Up Players")
    render_top_player_cards(squad_table, "passing_volume")

    st.markdown("#### Defensive Anchors")
    render_top_player_cards(squad_table, "defensive_volume")

    st.markdown("### Squad Table")

    squad_cols = [
        "line",
        "position",
        "player",
        "general_role",
        "offensive_role",
        "defensive_role",
        "importance_score",
        "importance_tier",
        "matches",
        "xg",
        "shots",
        "passes",
        "pressures",
        "recoveries",
        "duels",
        "strengths",
        "weaknesses"
    ]

    existing_cols = [c for c in squad_cols if c in squad_table.columns]

    st.dataframe(
        squad_table[existing_cols],
        use_container_width=True,
        hide_index=True,
        height=520
    )


# ============================================================
# SIMILARITY
# ============================================================

with tab_similarity:

    sim1, sim2 = st.columns([1, 1])

    with sim1:
        fig = plot_similarity_bar_safe(team_similarity, tournament_id, team, primary_color)

        if fig is not None:
            st.pyplot(fig, use_container_width=True)

    with sim2:
        fig = plot_team_radar_comparison(team_master, team_similarity, tournament_id, team)

        if fig is not None:
            st.pyplot(fig, use_container_width=True)

    st.markdown("### Similar Teams Table")

    sim = team_similarity[
        (team_similarity["tournament_id"] == tournament_id) &
        (team_similarity["team"] == team)
    ].copy()

    sim_cols = [
        "similar_team",
        "similar_tournament_id",
        "similarity_score",
        "similarity_rank"
    ]

    existing_cols = [c for c in sim_cols if c in sim.columns]

    st.dataframe(
        sim[existing_cols].sort_values("similarity_rank").head(10),
        use_container_width=True,
        hide_index=True
    )


# ============================================================
# FOOTER
# ============================================================

app_footer()
'''

page_path.write_text(team_tournament_code.strip(), encoding="utf-8")

print("1_Team_Tournament.py updated successfully")

1_Team_Tournament.py updated successfully


In [3]:
from pathlib import Path

pages_path = Path("streamlit_app/pages")
pages_path.mkdir(parents=True, exist_ok=True)

player_tournament_code = r'''
import ast
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import streamlit as st
from mplsoccer import Pitch

from utils.data_loader import load_all_data
from utils.filters import (
    get_tournament_options,
    get_team_options,
    get_player_options,
    filter_player
)
from utils.style import apply_global_style
from utils.cards import metric_card, text_card


st.set_page_config(page_title="Player Tournament", layout="wide")
apply_global_style()

data = load_all_data()

matches = data["matches"]
events = data["events"]
player_master = data["player_master"]
player_role_fit = data["player_role_fit"]
player_similarity = data["player_similarity"]
highlighted_matches = data["highlighted_matches"]
assets = data["assets"]

if "event_type" not in events.columns and "type" in events.columns:
    events["event_type"] = events["type"]

BASE_DIR = Path(__file__).resolve().parents[2]


# ============================================================
# HELPERS
# ============================================================

def parse_location(value):
    if isinstance(value, list):
        return value

    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            return None

    return None


def parse_list_value(value):
    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            pass

        if "," in value:
            return [v.strip() for v in value.split(",") if v.strip()]

        if value.strip():
            return [value.strip()]

    return []


def list_card(title, values):
    values = parse_list_value(values)

    if len(values) == 0:
        values = ["No data available"]

    html_items = "".join([f"<li>{v}</li>" for v in values])

    st.markdown(
        f"""
        <div class="custom-card">
            <div class="section-label">{title}</div>
            <ul style="margin-bottom:0; padding-left:18px; line-height:1.6;">
                {html_items}
            </ul>
        </div>
        """,
        unsafe_allow_html=True
    )


def ensure_xy(df):
    df = df.copy()

    if "x" not in df.columns:
        df["x"] = np.nan

    if "y" not in df.columns:
        df["y"] = np.nan

    if "location" in df.columns:
        loc = df["location"].apply(parse_location)

        df["x"] = df["x"].fillna(
            loc.apply(lambda v: v[0] if isinstance(v, list) and len(v) > 0 else np.nan)
        )

        df["y"] = df["y"].fillna(
            loc.apply(lambda v: v[1] if isinstance(v, list) and len(v) > 1 else np.nan)
        )

    return df


def draw_pitch():
    pitch = Pitch(
        pitch_type="statsbomb",
        pitch_color="#F8FAFC",
        line_color="#111827",
        linewidth=1.1
    )

    fig, ax = pitch.draw(figsize=(10, 7))
    return pitch, fig, ax


def get_team_asset(team_name):
    asset_row = assets[assets["team"] == team_name]

    if len(asset_row) == 0:
        return None, "#2563EB"

    primary_color = asset_row.iloc[0].get("primary_color", "#2563EB")
    raw_flag_path = asset_row.iloc[0].get("flag_path", None)

    if not raw_flag_path or pd.isna(raw_flag_path):
        return None, primary_color

    raw_flag_path = str(raw_flag_path)

    possible_paths = [
        BASE_DIR / raw_flag_path,
        BASE_DIR / "streamlit_app" / raw_flag_path,
        Path(raw_flag_path)
    ]

    for p in possible_paths:
        if p.exists():
            return str(p), primary_color

    return None, primary_color


def safe_value(row, col, default=0):
    if row is None:
        return default

    if col not in row.index:
        return default

    value = row.get(col, default)

    if pd.isna(value):
        return default

    return value


def get_row(df):
    if len(df) == 0:
        return None

    return df.iloc[0]


def compact_table(df, preferred_cols):
    existing_cols = [c for c in preferred_cols if c in df.columns]

    if len(existing_cols) == 0:
        return df

    return df[existing_cols]


def pct(numerator, denominator):
    numerator = 0 if pd.isna(numerator) else numerator
    denominator = 0 if pd.isna(denominator) else denominator

    if denominator == 0:
        return 0

    return round((numerator / denominator) * 100, 1)


def get_first_available(row, cols, default=0):
    if row is None:
        return default

    for col in cols:
        if col in row.index and pd.notna(row.get(col)):
            return row.get(col)

    return default


def player_events_filter(events, tournament_id, team, player):
    df = events[
        (events["tournament_id"] == tournament_id) &
        (events["team"] == team) &
        (events["player"] == player)
    ].copy()

    if "period" in df.columns:
        df = df[df["period"].fillna(0).astype(float) <= 4].copy()

    return df


def build_player_event_kpis(player_events):
    df = player_events.copy()

    shots = len(df[df["event_type"] == "Shot"])
    goals = 0

    if "shot_outcome" in df.columns:
        goals = len(
            df[
                (df["event_type"] == "Shot") &
                (df["shot_outcome"] == "Goal")
            ]
        )

    passes = len(df[df["event_type"] == "Pass"])
    completed_passes = 0

    if "pass_outcome" in df.columns:
        completed_passes = len(
            df[
                (df["event_type"] == "Pass") &
                (df["pass_outcome"].isna())
            ]
        )

    duels = len(df[df["event_type"] == "Duel"])
    duels_won = 0

    if "duel_outcome" in df.columns:
        duels_won = len(
            df[
                (df["event_type"] == "Duel") &
                (df["duel_outcome"].astype(str).str.contains("Won|Success", case=False, na=False))
            ]
        )

    pressures = len(df[df["event_type"] == "Pressure"])
    recoveries = len(df[df["event_type"].isin(["Ball Recovery", "Interception"])])
    carries = len(df[df["event_type"] == "Carry"])

    crosses = 0
    completed_crosses = 0

    if "pass_cross" in df.columns:
        cross_df = df[
            (df["event_type"] == "Pass") &
            (df["pass_cross"].fillna(False) == True)
        ]

        crosses = len(cross_df)

        if "pass_outcome" in cross_df.columns:
            completed_crosses = len(cross_df[cross_df["pass_outcome"].isna()])

    shot_on_target = 0

    if "shot_outcome" in df.columns:
        shot_on_target = len(
            df[
                (df["event_type"] == "Shot") &
                (df["shot_outcome"].astype(str).isin(["Goal", "Saved", "Saved To Post"]))
            ]
        )

    xg = 0
    if "shot_statsbomb_xg" in df.columns:
        xg = df["shot_statsbomb_xg"].fillna(0).sum()

    return {
        "shots": shots,
        "goals": goals,
        "xg": xg,
        "shot_accuracy_pct": pct(shot_on_target, shots),
        "goal_conversion_pct": pct(goals, shots),
        "passes": passes,
        "completed_passes": completed_passes,
        "pass_completion_pct": pct(completed_passes, passes),
        "duels": duels,
        "duels_won": duels_won,
        "duel_win_pct": pct(duels_won, duels),
        "crosses": crosses,
        "completed_crosses": completed_crosses,
        "cross_completion_pct": pct(completed_crosses, crosses),
        "pressures": pressures,
        "recoveries": recoveries,
        "carries": carries,
    }


def build_player_dimension_scores(row, event_kpis):
    if row is None:
        return pd.DataFrame()

    attack = get_first_available(row, ["attacking_volume", "attack_score", "xg", "shots"], event_kpis["shots"])
    passing = get_first_available(row, ["passing_volume", "passes", "build_up_score"], event_kpis["passes"])
    carrying = get_first_available(row, ["carrying_volume", "carries", "progressive_actions"], event_kpis["carries"])
    defense = get_first_available(row, ["defensive_volume", "defense_score", "recoveries", "duels"], event_kpis["recoveries"])
    pressing = get_first_available(row, ["pressing_volume", "pressures", "pressure_volume"], event_kpis["pressures"])
    importance = get_first_available(row, ["importance_score", "gk_importance_score", "outfield_importance_score"], 0)

    raw = pd.DataFrame({
        "dimension": ["Attack", "Passing", "Carrying", "Defense", "Pressing", "Importance"],
        "value": [attack, passing, carrying, defense, pressing, importance]
    })

    raw["value"] = pd.to_numeric(raw["value"], errors="coerce").fillna(0)

    max_value = raw["value"].max()

    if max_value > 100:
        raw["score"] = (raw["value"] / max_value) * 100
    else:
        raw["score"] = raw["value"]

    raw["score"] = raw["score"].clip(0, 100)

    return raw


def generate_player_summary(row, player, team, tournament_id, event_kpis):
    if row is None:
        return "No player profile available."

    position = safe_value(row, "position", "N/A")
    general_role = safe_value(row, "general_role", "N/A")
    offensive_role = safe_value(row, "offensive_role", "N/A")
    defensive_role = safe_value(row, "defensive_role", "N/A")
    importance = round(safe_value(row, "importance_score", 0), 2)
    tier = safe_value(row, "importance_tier", "N/A")

    return (
        f"{player} played mainly as {position} for {team} in {tournament_id}. "
        f"The player profiles as a {general_role}, with an offensive role of {offensive_role} "
        f"and a defensive role of {defensive_role}. "
        f"His importance score is {importance}, classified as {tier}. "
        f"Across the selected tournament data, he registered {event_kpis['shots']} shots, "
        f"{event_kpis['passes']} passes, {event_kpis['carries']} carries, "
        f"{event_kpis['pressures']} pressures and {event_kpis['recoveries']} recoveries/interceptions."
    )


# ============================================================
# VISUALS
# ============================================================

def plot_player_radar(row, event_kpis, player, team_color="#2563EB"):
    scores = build_player_dimension_scores(row, event_kpis)

    if len(scores) == 0:
        return None

    labels = scores["dimension"].tolist()
    values = scores["score"].astype(float).tolist()
    values += values[:1]

    angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(6.5, 6.5), subplot_kw=dict(polar=True))

    ax.plot(angles, values, linewidth=2.5, color=team_color)
    ax.fill(angles, values, color=team_color, alpha=0.15)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=10, weight="bold")
    ax.set_ylim(0, 100)

    ax.set_title(f"{player} | Profile Radar", fontsize=15, weight="bold", pad=20)

    return fig


def plot_player_profile_bars(row, event_kpis, player, team_color="#2563EB"):
    scores = build_player_dimension_scores(row, event_kpis)

    if len(scores) == 0:
        return None

    scores = scores.sort_values("score", ascending=True)

    fig, ax = plt.subplots(figsize=(8, 4))

    ax.barh(scores["dimension"], scores["score"], color=team_color, alpha=0.85)
    ax.set_xlim(0, 100)
    ax.set_title(f"{player} | Profile Scores", fontsize=15, weight="bold")
    ax.set_xlabel("Score")

    for i, value in enumerate(scores["score"]):
        ax.text(value + 1, i, f"{value:.1f}", va="center", fontsize=9, weight="bold")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    return fig


def plot_player_zone_heatmap(player_events, event_types, title, cmap="Blues"):
    df = player_events[player_events["event_type"].isin(event_types)].copy()

    df = ensure_xy(df)
    df = df.dropna(subset=["x", "y"])

    if len(df) == 0:
        return None

    pitch, fig, ax = draw_pitch()

    bin_statistic = pitch.bin_statistic(
        df["x"],
        df["y"],
        statistic="count",
        bins=(6, 4)
    )

    pitch.heatmap(bin_statistic, ax=ax, cmap=cmap, alpha=0.78)
    pitch.label_heatmap(
        bin_statistic,
        color="#111827",
        fontsize=10,
        ax=ax,
        ha="center",
        va="center"
    )

    ax.set_title(title, fontsize=15, weight="bold")

    return fig


def plot_shot_map(player_events, player, team_color="#2563EB"):
    df = player_events[player_events["event_type"] == "Shot"].copy()

    df = ensure_xy(df)
    df = df.dropna(subset=["x", "y"])

    if len(df) == 0:
        return None

    if "shot_statsbomb_xg" not in df.columns:
        df["shot_statsbomb_xg"] = 0

    df["shot_xg"] = df["shot_statsbomb_xg"].fillna(0)

    pitch, fig, ax = draw_pitch()

    goals = df[df["shot_outcome"] == "Goal"] if "shot_outcome" in df.columns else pd.DataFrame()
    shots = df[df["shot_outcome"] != "Goal"] if "shot_outcome" in df.columns else df

    if len(shots) > 0:
        pitch.scatter(
            shots["x"],
            shots["y"],
            s=shots["shot_xg"] * 1300 + 65,
            ax=ax,
            color=team_color,
            alpha=0.50,
            label="Shot"
        )

    if len(goals) > 0:
        pitch.scatter(
            goals["x"],
            goals["y"],
            s=goals["shot_xg"] * 1500 + 100,
            ax=ax,
            color="#DC2626",
            edgecolors="#111827",
            linewidth=1.5,
            label="Goal"
        )

    ax.set_title(
        f"{player} Shot Map | Shots {len(df)} | Goals {len(goals)} | xG {df['shot_xg'].sum():.2f}",
        fontsize=15,
        weight="bold"
    )

    ax.legend(loc="upper left")

    return fig


def plot_cross_origin_zones(player_events, player):
    df = player_events[player_events["event_type"] == "Pass"].copy()

    if "pass_cross" not in df.columns:
        return None

    df = df[df["pass_cross"].fillna(False) == True].copy()

    return plot_player_zone_heatmap(
        df,
        ["Pass"],
        f"{player} | Cross Origin Zones",
        cmap="Blues"
    )


def plot_reception_zones(events, tournament_id, team, player):
    if "pass_recipient" not in events.columns:
        return None

    df = events[
        (events["tournament_id"] == tournament_id) &
        (events["team"] == team) &
        (events["pass_recipient"] == player) &
        (events["event_type"] == "Pass")
    ].copy()

    if "pass_end_location" not in df.columns:
        return None

    pass_end = df["pass_end_location"].apply(parse_location)

    df["x"] = pass_end.apply(lambda v: v[0] if isinstance(v, list) and len(v) > 0 else np.nan)
    df["y"] = pass_end.apply(lambda v: v[1] if isinstance(v, list) and len(v) > 1 else np.nan)

    df = df.dropna(subset=["x", "y"])

    if len(df) == 0:
        return None

    pitch, fig, ax = draw_pitch()

    bin_statistic = pitch.bin_statistic(df["x"], df["y"], statistic="count", bins=(6, 4))
    pitch.heatmap(bin_statistic, ax=ax, cmap="Purples", alpha=0.78)
    pitch.label_heatmap(bin_statistic, color="#111827", fontsize=10, ax=ax)

    ax.set_title(f"{player} | Reception Zones", fontsize=15, weight="bold")

    return fig


def plot_progressive_actions(player_events, player, team_color="#2563EB"):
    df = player_events[player_events["event_type"].isin(["Pass", "Carry"])].copy()

    if len(df) == 0:
        return None

    df = ensure_xy(df)

    df["end_x"] = np.nan
    df["end_y"] = np.nan

    if "pass_end_location" in df.columns:
        pass_end = df["pass_end_location"].apply(parse_location)
        df["pass_end_x"] = pass_end.apply(lambda v: v[0] if isinstance(v, list) and len(v) > 0 else np.nan)
        df["pass_end_y"] = pass_end.apply(lambda v: v[1] if isinstance(v, list) and len(v) > 1 else np.nan)
        df["end_x"] = df["end_x"].fillna(df["pass_end_x"])
        df["end_y"] = df["end_y"].fillna(df["pass_end_y"])

    if "carry_end_location" in df.columns:
        carry_end = df["carry_end_location"].apply(parse_location)
        df["carry_end_x"] = carry_end.apply(lambda v: v[0] if isinstance(v, list) and len(v) > 0 else np.nan)
        df["carry_end_y"] = carry_end.apply(lambda v: v[1] if isinstance(v, list) and len(v) > 1 else np.nan)
        df["end_x"] = df["end_x"].fillna(df["carry_end_x"])
        df["end_y"] = df["end_y"].fillna(df["carry_end_y"])

    df = df.dropna(subset=["x", "y", "end_x", "end_y"])
    df["progression"] = df["end_x"] - df["x"]
    df = df[df["progression"] >= 15].copy()

    if len(df) == 0:
        return None

    pitch, fig, ax = draw_pitch()

    passes = df[df["event_type"] == "Pass"]
    carries = df[df["event_type"] == "Carry"]

    if len(passes) > 0:
        pitch.arrows(
            passes["x"],
            passes["y"],
            passes["end_x"],
            passes["end_y"],
            ax=ax,
            width=2,
            headwidth=4,
            color=team_color,
            alpha=0.55,
            label="Progressive Pass"
        )

    if len(carries) > 0:
        pitch.arrows(
            carries["x"],
            carries["y"],
            carries["end_x"],
            carries["end_y"],
            ax=ax,
            width=2,
            headwidth=4,
            color="#111827",
            alpha=0.55,
            label="Progressive Carry"
        )

    ax.set_title(f"{player} | Progressive Actions", fontsize=15, weight="bold")
    ax.legend(loc="upper left")

    return fig


def plot_role_fit_bar(rf, player, team_color="#2563EB"):
    if len(rf) == 0:
        return None

    df = rf.copy()

    role_col = next((c for c in ["target_role", "role", "role_name"] if c in df.columns), None)
    score_col = next((c for c in ["role_fit_score", "fit_score", "score"] if c in df.columns), None)

    if role_col is None or score_col is None:
        return None

    df[score_col] = pd.to_numeric(df[score_col], errors="coerce").fillna(0)
    df = df.sort_values(score_col, ascending=False).head(8)

    fig, ax = plt.subplots(figsize=(8, 4))

    ax.barh(df[role_col][::-1], df[score_col][::-1], color=team_color, alpha=0.85)
    ax.set_title(f"{player} | Role Fit", fontsize=15, weight="bold")
    ax.set_xlabel("Role Fit Score")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    return fig


def plot_similarity_bar(sim, player, team_color="#2563EB"):
    if len(sim) == 0:
        return None

    df = sim.copy()

    player_col = next((c for c in ["similar_player", "player_2", "similar_name"] if c in df.columns), None)
    score_col = next((c for c in ["similarity_score", "score"] if c in df.columns), None)

    if player_col is None:
        return None

    if score_col is None:
        return None

    df[score_col] = pd.to_numeric(df[score_col], errors="coerce").fillna(0)

    if "similarity_rank" in df.columns:
        df = df.sort_values("similarity_rank")
    else:
        df = df.sort_values(score_col, ascending=False)

    df = df.head(8)

    labels = df[player_col].astype(str)

    if "similar_team" in df.columns:
        labels = labels + " | " + df["similar_team"].astype(str)

    if "similar_tournament_id" in df.columns:
        labels = labels + " | " + df["similar_tournament_id"].astype(str)

    fig, ax = plt.subplots(figsize=(8, 4))

    ax.barh(labels[::-1], df[score_col][::-1], color=team_color, alpha=0.85)

    ax.set_title(f"{player} | Similar Players", fontsize=15, weight="bold")
    ax.set_xlabel("Similarity")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    return fig


def plot_highlighted_matches_bar(hm, player, team_color="#2563EB"):
    if len(hm) == 0:
        return None

    df = hm.copy()

    score_col = next((c for c in ["match_score", "player_match_score", "score", "importance_score"] if c in df.columns), None)
    label_col = next((c for c in ["match_label", "opponent", "match_name"] if c in df.columns), None)

    if score_col is None:
        return None

    if label_col is None:
        if "match_id" in df.columns:
            label_col = "match_id"
        else:
            return None

    df[score_col] = pd.to_numeric(df[score_col], errors="coerce").fillna(0)

    if "match_score_rank" in df.columns:
        df = df.sort_values("match_score_rank")
    else:
        df = df.sort_values(score_col, ascending=False)

    df = df.head(8)

    fig, ax = plt.subplots(figsize=(8, 4))

    ax.barh(df[label_col].astype(str)[::-1], df[score_col][::-1], color=team_color, alpha=0.85)
    ax.set_title(f"{player} | Highlighted Matches", fontsize=15, weight="bold")
    ax.set_xlabel("Match Score")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    return fig


# ============================================================
# SIDEBAR
# ============================================================

st.sidebar.title("Filters")

tournament_id = st.sidebar.selectbox("Tournament", get_tournament_options(matches))
team = st.sidebar.selectbox("Team", get_team_options(player_master, tournament_id))
player = st.sidebar.selectbox("Player", get_player_options(player_master, tournament_id, team))

profile = filter_player(player_master, tournament_id, team, player)
row = get_row(profile)

flag_path, primary_color = get_team_asset(team)

player_events = player_events_filter(events, tournament_id, team, player)
event_kpis = build_player_event_kpis(player_events)


# ============================================================
# FILTERED DATA
# ============================================================

rf = player_role_fit[
    (player_role_fit["tournament_id"] == tournament_id) &
    (player_role_fit["team"] == team) &
    (player_role_fit["player"] == player)
].copy()

sim = player_similarity[
    (player_similarity["tournament_id"] == tournament_id) &
    (player_similarity["team"] == team) &
    (player_similarity["player"] == player)
].copy()

hm = highlighted_matches[
    (highlighted_matches["tournament_id"] == tournament_id) &
    (highlighted_matches["team"] == team) &
    (highlighted_matches["player"] == player)
].copy()

if "level" in hm.columns:
    hm = hm[hm["level"] == "player"].copy()


# ============================================================
# HEADER
# ============================================================

col_flag, col_title = st.columns([1, 6])

with col_flag:
    if flag_path:
        st.image(flag_path, width=85)

with col_title:
    st.markdown(
        f"<h1 style='color:{primary_color}; margin-bottom:0;'>{player}</h1>",
        unsafe_allow_html=True
    )
    st.markdown(
        f"<div class='page-subtitle'>{team} | {tournament_id} | Player Tournament Analysis</div>",
        unsafe_allow_html=True
    )


# ============================================================
# MAIN KPI ROW
# ============================================================

if row is not None:

    k1, k2, k3, k4, k5, k6 = st.columns(6)

    with k1:
        metric_card("Matches", int(safe_value(row, "matches", 0)))

    with k2:
        metric_card("Position", safe_value(row, "position", "N/A"))

    with k3:
        metric_card("Role", safe_value(row, "general_role", "N/A"))

    with k4:
        metric_card("Importance", round(safe_value(row, "importance_score", 0), 2))

    with k5:
        metric_card("Tier", safe_value(row, "importance_tier", "N/A"))

    with k6:
        metric_card("xG", round(safe_value(row, "xg", event_kpis["xg"]), 2))


# ============================================================
# EFFICIENCY KPI ROW
# ============================================================

e1, e2, e3, e4, e5, e6 = st.columns(6)

with e1:
    metric_card("Shots", event_kpis["shots"])

with e2:
    metric_card("Goals", event_kpis["goals"])

with e3:
    metric_card("Shot Acc. %", f"{event_kpis['shot_accuracy_pct']}%")

with e4:
    metric_card("Pass Acc. %", f"{event_kpis['pass_completion_pct']}%")

with e5:
    metric_card("Duel Win %", f"{event_kpis['duel_win_pct']}%")

with e6:
    metric_card("Recoveries", event_kpis["recoveries"])


# ============================================================
# TABS
# ============================================================

tab_overview, tab_attack, tab_organization, tab_defense, tab_matches, tab_similarity = st.tabs([
    "Overview",
    "Attack",
    "Organization",
    "Defense",
    "Matches",
    "Role & Similarity"
])


# ============================================================
# OVERVIEW
# ============================================================

with tab_overview:

    st.markdown("## Player Overview")

    if row is not None:

        text_card(
            "Auto Player Summary",
            generate_player_summary(row, player, team, tournament_id, event_kpis)
        )

        info_col, radar_col, bars_col = st.columns([1, 1, 1])

        with info_col:
            text_card("General Role", safe_value(row, "general_role", "N/A"))
            text_card("Offensive Role", safe_value(row, "offensive_role", "N/A"))
            text_card("Defensive Role", safe_value(row, "defensive_role", "N/A"))

            list_card("Strengths", safe_value(row, "strengths", []))
            list_card("Weaknesses", safe_value(row, "weaknesses", []))

            if "best_role_fits" in row.index:
                text_card("Best Role Fits", safe_value(row, "best_role_fits", "N/A"))

        with radar_col:
            fig = plot_player_radar(row, event_kpis, player, team_color=primary_color)

            if fig is not None:
                st.pyplot(fig, use_container_width=True)

        with bars_col:
            fig = plot_player_profile_bars(row, event_kpis, player, team_color=primary_color)

            if fig is not None:
                st.pyplot(fig, use_container_width=True)

    else:
        st.info("No player profile available.")


# ============================================================
# ATTACK
# ============================================================

with tab_attack:

    st.markdown("## Attack")

    a1, a2, a3, a4, a5 = st.columns(5)

    with a1:
        metric_card("Goals", event_kpis["goals"])

    with a2:
        metric_card("Shots", event_kpis["shots"])

    with a3:
        metric_card("xG", round(safe_value(row, "xg", event_kpis["xg"]), 2))

    with a4:
        metric_card("Shot Acc. %", f"{event_kpis['shot_accuracy_pct']}%")

    with a5:
        metric_card("Crosses", event_kpis["crosses"])

    st.markdown("### Shooting")

    s1, s2 = st.columns(2)

    with s1:
        fig = plot_shot_map(player_events, player, team_color=primary_color)

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No shot map available.")

    with s2:
        fig = plot_player_zone_heatmap(
            player_events,
            ["Shot"],
            f"{player} | Shot Zones",
            cmap="Reds"
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No shot zone data available.")

    st.markdown("### Crossing & Offensive Duels")

    c1, c2 = st.columns(2)

    with c1:
        fig = plot_cross_origin_zones(player_events, player)

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No cross origin data available.")

    with c2:
        offensive_duels = ensure_xy(player_events)
        offensive_duels = offensive_duels[offensive_duels["x"].fillna(0) >= 60].copy()

        fig = plot_player_zone_heatmap(
            offensive_duels,
            ["Duel"],
            f"{player} | Offensive Duel Zones",
            cmap="Oranges"
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No offensive duel data available.")


# ============================================================
# ORGANIZATION
# ============================================================

with tab_organization:

    st.markdown("## Organization & Build-Up")

    o1, o2, o3, o4 = st.columns(4)

    with o1:
        metric_card("Passes", event_kpis["passes"])

    with o2:
        metric_card("Pass Acc. %", f"{event_kpis['pass_completion_pct']}%")

    with o3:
        metric_card("Carries", event_kpis["carries"])

    with o4:
        metric_card("Progression", round(safe_value(row, "progressive_actions", 0), 1))

    z1, z2 = st.columns(2)

    with z1:
        fig = plot_player_zone_heatmap(
            player_events,
            ["Pass"],
            f"{player} | Pass Origin Zones",
            cmap="Blues"
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No pass zone data available.")

    with z2:
        fig = plot_player_zone_heatmap(
            player_events,
            ["Carry"],
            f"{player} | Carry Zones",
            cmap="Purples"
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No carry zone data available.")

    st.markdown("### Progressive Actions & Receptions")

    p1, p2 = st.columns(2)

    with p1:
        fig = plot_progressive_actions(player_events, player, team_color=primary_color)

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No progressive actions available.")

    with p2:
        fig = plot_reception_zones(events, tournament_id, team, player)

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No reception zones available.")


# ============================================================
# DEFENSE
# ============================================================

with tab_defense:

    st.markdown("## Defense")

    d1, d2, d3, d4 = st.columns(4)

    with d1:
        metric_card("Pressures", event_kpis["pressures"])

    with d2:
        metric_card("Recoveries", event_kpis["recoveries"])

    with d3:
        metric_card("Duels", event_kpis["duels"])

    with d4:
        metric_card("Duel Win %", f"{event_kpis['duel_win_pct']}%")

    z1, z2 = st.columns(2)

    with z1:
        fig = plot_player_zone_heatmap(
            player_events,
            ["Pressure"],
            f"{player} | Pressure Zones",
            cmap="Oranges"
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No pressure zone data available.")

    with z2:
        fig = plot_player_zone_heatmap(
            player_events,
            ["Ball Recovery", "Interception"],
            f"{player} | Recovery & Interception Zones",
            cmap="Greens"
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No recovery zone data available.")

    st.markdown("### Duel Zones")

    fig = plot_player_zone_heatmap(
        player_events,
        ["Duel"],
        f"{player} | Duel Zones",
        cmap="Purples"
    )

    if fig is not None:
        st.pyplot(fig, use_container_width=True)
    else:
        st.info("No duel zone data available.")


# ============================================================
# MATCHES
# ============================================================

with tab_matches:

    st.markdown("## Highlighted Matches")

    if len(hm) > 0:

        if "match_score_rank" in hm.columns:
            hm = hm.sort_values("match_score_rank")

        fig = plot_highlighted_matches_bar(hm, player, team_color=primary_color)

        if fig is not None:
            st.pyplot(fig, use_container_width=True)

        preferred_cols = [
            "match_score_rank",
            "match_label",
            "opponent",
            "match_score",
            "minutes",
            "xg",
            "shots",
            "passes",
            "carries",
            "pressures",
            "recoveries",
            "duels"
        ]

        st.dataframe(
            compact_table(hm, preferred_cols).head(10),
            use_container_width=True,
            hide_index=True
        )

    else:
        st.info("No highlighted matches available.")


# ============================================================
# ROLE & SIMILARITY
# ============================================================

with tab_similarity:

    s1, s2 = st.columns(2)

    with s1:
        st.markdown("## Role Fit")

        if len(rf) > 0:
            if "role_fit_score" in rf.columns:
                rf = rf.sort_values("role_fit_score", ascending=False)

            fig = plot_role_fit_bar(rf, player, team_color=primary_color)

            if fig is not None:
                st.pyplot(fig, use_container_width=True)

            st.dataframe(
                compact_table(
                    rf,
                    [
                        "target_role",
                        "role_fit_score",
                        "role_fit_rank",
                        "current_role",
                        "position"
                    ]
                ),
                use_container_width=True,
                hide_index=True
            )
        else:
            st.info("No role fit data available.")

    with s2:
        st.markdown("## Similar Players")

        if len(sim) > 0:
            if "similarity_rank" in sim.columns:
                sim = sim.sort_values("similarity_rank")

            fig = plot_similarity_bar(sim, player, team_color=primary_color)

            if fig is not None:
                st.pyplot(fig, use_container_width=True)

            st.dataframe(
                compact_table(
                    sim,
                    [
                        "similarity_rank",
                        "similar_player",
                        "similar_team",
                        "similar_tournament_id",
                        "similar_position",
                        "similar_general_role",
                        "similarity_score"
                    ]
                ).head(10),
                use_container_width=True,
                hide_index=True
            )
        else:
            st.info("No similar players available.")


# ============================================================
# FOOTER
# ============================================================

st.divider()
st.caption(
    "Developed by Juan Ignacio Breccia | Football Analytics · Scouting · Data Science | Powered by StatsBomb Open Data"
)
'''

with open(pages_path / "4_Player_Tournament.py", "w", encoding="utf-8") as f:
    f.write(player_tournament_code.strip())

print("4_Player_Tournament.py updated successfully")

4_Player_Tournament.py updated successfully


In [5]:
from pathlib import Path

pages_path = Path("streamlit_app/pages")
pages_path.mkdir(parents=True, exist_ok=True)

team_match_code = r'''
import ast
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import streamlit as st
from mplsoccer import Pitch

from utils.data_loader import load_all_data
from utils.style import apply_global_style
from utils.cards import metric_card, text_card


st.set_page_config(page_title="Team Match", layout="wide")
apply_global_style()

data = load_all_data()

matches = data["matches"]
events = data["events"]
assets = data.get("assets", pd.DataFrame())

if "event_type" not in events.columns and "type" in events.columns:
    events["event_type"] = events["type"]

if "type" not in events.columns and "event_type" in events.columns:
    events["type"] = events["event_type"]


BASE_DIR = Path(__file__).resolve().parents[2]


POSITION_ORDER = {
    "Goalkeeper": 1,
    "Right Back": 2,
    "Right Center Back": 3,
    "Center Back": 4,
    "Left Center Back": 5,
    "Left Back": 6,
    "Right Wing Back": 7,
    "Left Wing Back": 8,
    "Right Defensive Midfield": 9,
    "Center Defensive Midfield": 10,
    "Left Defensive Midfield": 11,
    "Right Center Midfield": 12,
    "Center Midfield": 13,
    "Left Center Midfield": 14,
    "Right Midfield": 15,
    "Left Midfield": 16,
    "Right Attacking Midfield": 17,
    "Center Attacking Midfield": 18,
    "Left Attacking Midfield": 19,
    "Right Wing": 20,
    "Left Wing": 21,
    "Right Center Forward": 22,
    "Center Forward": 23,
    "Left Center Forward": 24,
    "Secondary Striker": 25
}


def get_col(df, options, default=None):
    for col in options:
        if col in df.columns:
            return col
    return default


def get_row_value(row, options, default=""):
    for col in options:
        if col in row.index and pd.notna(row[col]):
            return row[col]
    return default


def parse_location(value):
    if isinstance(value, list):
        return value

    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            return None

    return None


def parse_any(value):
    if isinstance(value, (list, dict)):
        return value

    if isinstance(value, str):
        try:
            return ast.literal_eval(value)
        except Exception:
            return None

    return None


def ensure_xy(df):
    df = df.copy()

    if "x" not in df.columns:
        df["x"] = np.nan

    if "y" not in df.columns:
        df["y"] = np.nan

    if "location" in df.columns:
        loc = df["location"].apply(parse_location)

        df["x"] = df["x"].fillna(
            loc.apply(lambda v: v[0] if isinstance(v, list) and len(v) > 0 else np.nan)
        )

        df["y"] = df["y"].fillna(
            loc.apply(lambda v: v[1] if isinstance(v, list) and len(v) > 1 else np.nan)
        )

    return df


def draw_pitch():
    pitch = Pitch(
        pitch_type="statsbomb",
        pitch_color="#F8FAFC",
        line_color="#111827",
        linewidth=1.1
    )

    fig, ax = pitch.draw(figsize=(10, 7))
    return pitch, fig, ax


def clean_player_name(name):
    parts = str(name).split()
    return parts[-1] if len(parts) > 1 else str(name)


def get_team_asset(team_name):
    if assets is None or len(assets) == 0 or "team" not in assets.columns:
        return None, "#2563EB"

    asset_row = assets[assets["team"] == team_name]

    if len(asset_row) == 0:
        return None, "#2563EB"

    primary_color = asset_row.iloc[0].get("primary_color", "#2563EB")
    raw_flag_path = asset_row.iloc[0].get("flag_path", None)

    if not raw_flag_path or pd.isna(raw_flag_path):
        return None, primary_color

    raw_flag_path = str(raw_flag_path)

    possible_paths = [
        BASE_DIR / raw_flag_path,
        BASE_DIR / "streamlit_app" / raw_flag_path,
        Path(raw_flag_path)
    ]

    for path in possible_paths:
        if path.exists():
            return str(path), primary_color

    return None, primary_color


def build_match_label(df):
    date_col = get_col(df, ["match_date", "date"], default=None)

    if date_col is None:
        date_part = "Match"
    else:
        date_part = df[date_col].astype(str)

    return (
        date_part
        + " | "
        + df["home_team"].astype(str)
        + " "
        + df["home_score"].astype(str)
        + "-"
        + df["away_score"].astype(str)
        + " "
        + df["away_team"].astype(str)
    )


def remove_penalty_shootout(df):
    df = df.copy()

    if "period" in df.columns:
        df = df[df["period"].fillna(0).astype(float) <= 4].copy()

    return df


def get_lineup_df(match_events, match_id, team):
    xi_events = match_events[
        (match_events["match_id"] == match_id) &
        (match_events["team"] == team) &
        (match_events["type"].astype(str).str.lower().eq("starting xi"))
    ].copy()

    rows = []

    if len(xi_events) == 0:
        return pd.DataFrame(columns=["player", "position", "position_order"])

    row = xi_events.iloc[0]
    parsed = None

    if "tactics_lineup" in row.index:
        parsed = parse_any(row["tactics_lineup"])

    if parsed is None and "tactics" in row.index:
        tactics = parse_any(row["tactics"])
        if isinstance(tactics, dict):
            parsed = tactics.get("lineup", None)

    if isinstance(parsed, dict) and "lineup" in parsed:
        parsed = parsed["lineup"]

    if isinstance(parsed, list):
        for item in parsed:
            if isinstance(item, dict):
                player_data = item.get("player", {})
                position_data = item.get("position", {})

                player_name = player_data.get("name", None) if isinstance(player_data, dict) else None
                position_name = position_data.get("name", "") if isinstance(position_data, dict) else ""

                if player_name:
                    rows.append({
                        "player": player_name,
                        "position": position_name,
                        "position_order": POSITION_ORDER.get(position_name, 99)
                    })

    return pd.DataFrame(rows)


def build_player_match_table(team_events):
    if len(team_events) == 0 or "player" not in team_events.columns:
        return pd.DataFrame()

    team_events = team_events.copy()

    if "shot_statsbomb_xg" not in team_events.columns:
        team_events["shot_statsbomb_xg"] = 0

    player_match = (
        team_events
        .dropna(subset=["player"])
        .groupby("player")
        .agg(
            events=("event_type", "count"),
            shots=("event_type", lambda x: (x == "Shot").sum()),
            passes=("event_type", lambda x: (x == "Pass").sum()),
            carries=("event_type", lambda x: (x == "Carry").sum()),
            recoveries=("event_type", lambda x: x.isin(["Ball Recovery", "Interception"]).sum()),
            pressures=("event_type", lambda x: (x == "Pressure").sum()),
            duels=("event_type", lambda x: (x == "Duel").sum()),
            fouls=("event_type", lambda x: (x == "Foul Committed").sum()),
            xg=("shot_statsbomb_xg", "sum")
        )
        .reset_index()
    )

    player_match["impact_score"] = (
        player_match["xg"] * 15
        + player_match["shots"] * 2
        + player_match["passes"] * 0.15
        + player_match["carries"] * 0.25
        + player_match["recoveries"] * 1.2
        + player_match["pressures"] * 0.6
        + player_match["duels"] * 0.5
    )

    return player_match.sort_values("impact_score", ascending=False)


def add_progressive_metrics_to_players(team_events, player_match):
    if len(player_match) == 0:
        return player_match

    df = team_events[team_events["type"].isin(["Pass", "Carry"])].copy()

    if len(df) == 0:
        player_match["progressive_actions"] = 0
        return player_match

    df = ensure_xy(df)
    df["end_x"] = np.nan

    if "pass_end_location" in df.columns:
        pass_end = df["pass_end_location"].apply(parse_location)
        df["pass_end_x"] = pass_end.apply(lambda v: v[0] if isinstance(v, list) and len(v) > 0 else np.nan)
        df["end_x"] = df["end_x"].fillna(df["pass_end_x"])

    if "carry_end_location" in df.columns:
        carry_end = df["carry_end_location"].apply(parse_location)
        df["carry_end_x"] = carry_end.apply(lambda v: v[0] if isinstance(v, list) and len(v) > 0 else np.nan)
        df["end_x"] = df["end_x"].fillna(df["carry_end_x"])

    df = df.dropna(subset=["player", "x", "end_x"])
    df["progression"] = df["end_x"] - df["x"]

    progressive = (
        df[df["progression"] >= 15]
        .groupby("player")
        .size()
        .reset_index(name="progressive_actions")
    )

    player_match = player_match.merge(progressive, on="player", how="left")
    player_match["progressive_actions"] = player_match["progressive_actions"].fillna(0)

    return player_match


def plot_match_xi_comparison(match_events, match_id, home_team, away_team, home_color="#2563EB", away_color="#DC2626"):
    home_xi = get_lineup_df(match_events, match_id, home_team)
    away_xi = get_lineup_df(match_events, match_id, away_team)

    if len(home_xi) == 0 and len(away_xi) == 0:
        return None

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.axis("off")

    teams_data = [
        (home_team, home_xi, home_color, 0.25),
        (away_team, away_xi, away_color, 0.75)
    ]

    for team_name, xi_df, color, x in teams_data:
        ax.text(x, 0.95, team_name, ha="center", va="top", fontsize=15, weight="bold", color=color)

        if len(xi_df) == 0:
            ax.text(x, 0.50, "No Starting XI available", ha="center", va="center", fontsize=11)
        else:
            xi_df = xi_df.sort_values("position_order").copy()

            for i, (_, p) in enumerate(xi_df.head(11).iterrows()):
                text = f"{i + 1}. {clean_player_name(p['player'])} · {p['position']}"
                ax.text(x, 0.86 - i * 0.07, text, ha="center", va="center", fontsize=10)

    ax.set_title("Starting XI Comparison", fontsize=16, weight="bold")

    return fig


def plot_match_xg_flow(match_events, match_id, team, opponent, team_color="#2563EB", opponent_color="#DC2626"):
    df = match_events[
        (match_events["match_id"] == match_id) &
        (match_events["type"] == "Shot")
    ].copy()

    df = remove_penalty_shootout(df)

    if len(df) == 0 or "minute" not in df.columns or "shot_statsbomb_xg" not in df.columns:
        return None

    fig, ax = plt.subplots(figsize=(10, 4.8))

    for team_name, color in [(team, team_color), (opponent, opponent_color)]:
        shots_df = df[df["team"] == team_name].copy()

        if len(shots_df) == 0:
            continue

        shots_df["shot_xg"] = shots_df["shot_statsbomb_xg"].fillna(0)
        shots_df = shots_df.sort_values("minute")
        shots_df["cumulative_xg"] = shots_df["shot_xg"].cumsum()

        ax.step(shots_df["minute"], shots_df["cumulative_xg"], where="post", linewidth=2.5, color=color, label=team_name)
        ax.scatter(shots_df["minute"], shots_df["cumulative_xg"], s=45, color=color)

    ax.set_title(f"xG Flow | {team} vs {opponent}", fontsize=15, weight="bold")
    ax.set_xlabel("Minute")
    ax.set_ylabel("Cumulative xG")
    ax.set_xlim(0, 120)
    ax.legend(loc="upper left")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    return fig


def plot_match_shot_map(match_events, match_id, team, team_color="#2563EB"):
    df = match_events[
        (match_events["match_id"] == match_id) &
        (match_events["team"] == team) &
        (match_events["type"] == "Shot")
    ].copy()

    df = remove_penalty_shootout(df)
    df = ensure_xy(df)
    df = df.dropna(subset=["x", "y"])

    if len(df) == 0:
        return None

    if "shot_statsbomb_xg" not in df.columns:
        df["shot_statsbomb_xg"] = 0

    outcome_col = get_col(df, ["shot_outcome", "shot_outcome_name"], default=None)

    df["shot_xg"] = df["shot_statsbomb_xg"].fillna(0)

    if outcome_col:
        goals = df[df[outcome_col] == "Goal"]
        shots_df = df[df[outcome_col] != "Goal"]
    else:
        goals = pd.DataFrame()
        shots_df = df

    pitch, fig, ax = draw_pitch()

    if len(shots_df) > 0:
        pitch.scatter(
            shots_df["x"],
            shots_df["y"],
            s=shots_df["shot_xg"] * 1300 + 45,
            ax=ax,
            color=team_color,
            alpha=0.45,
            label="Shot"
        )

    if len(goals) > 0:
        pitch.scatter(
            goals["x"],
            goals["y"],
            s=goals["shot_xg"] * 1500 + 90,
            ax=ax,
            color="#DC2626",
            edgecolors="#111827",
            linewidth=1.5,
            label="Goal"
        )

    ax.set_title(
        f"{team} Shot Map | Shots {len(df)} | Goals {len(goals)} | xG {df['shot_xg'].sum():.2f}",
        fontsize=15,
        weight="bold"
    )

    ax.legend(loc="upper left")
    return fig


def plot_match_zone_heatmap(match_events, match_id, team, event_types, title, cmap="Blues"):
    df = match_events[
        (match_events["match_id"] == match_id) &
        (match_events["team"] == team) &
        (match_events["type"].isin(event_types))
    ].copy()

    df = remove_penalty_shootout(df)
    df = ensure_xy(df)
    df = df.dropna(subset=["x", "y"])

    if len(df) == 0:
        return None

    pitch, fig, ax = draw_pitch()

    bin_statistic = pitch.bin_statistic(df["x"], df["y"], statistic="count", bins=(6, 4))
    pitch.heatmap(bin_statistic, ax=ax, cmap=cmap, alpha=0.78)
    pitch.label_heatmap(bin_statistic, color="#111827", fontsize=10, ax=ax, ha="center", va="center")

    ax.set_title(title, fontsize=15, weight="bold")
    return fig


def plot_box_entries(match_events, match_id, team, team_color="#2563EB"):
    df = match_events[
        (match_events["match_id"] == match_id) &
        (match_events["team"] == team) &
        (match_events["type"].isin(["Pass", "Carry"]))
    ].copy()

    df = remove_penalty_shootout(df)
    df = ensure_xy(df)

    df["end_x"] = np.nan
    df["end_y"] = np.nan

    if "pass_end_location" in df.columns:
        pass_end = df["pass_end_location"].apply(parse_location)
        df["pass_end_x"] = pass_end.apply(lambda v: v[0] if isinstance(v, list) and len(v) > 0 else np.nan)
        df["pass_end_y"] = pass_end.apply(lambda v: v[1] if isinstance(v, list) and len(v) > 1 else np.nan)
        df["end_x"] = df["end_x"].fillna(df["pass_end_x"])
        df["end_y"] = df["end_y"].fillna(df["pass_end_y"])

    if "carry_end_location" in df.columns:
        carry_end = df["carry_end_location"].apply(parse_location)
        df["carry_end_x"] = carry_end.apply(lambda v: v[0] if isinstance(v, list) and len(v) > 0 else np.nan)
        df["carry_end_y"] = carry_end.apply(lambda v: v[1] if isinstance(v, list) and len(v) > 1 else np.nan)
        df["end_x"] = df["end_x"].fillna(df["carry_end_x"])
        df["end_y"] = df["end_y"].fillna(df["carry_end_y"])

    df = df.dropna(subset=["x", "y", "end_x", "end_y"])

    entries = df[
        (df["end_x"] >= 102) &
        (df["end_y"] >= 18) &
        (df["end_y"] <= 62) &
        ~(
            (df["x"] >= 102) &
            (df["y"] >= 18) &
            (df["y"] <= 62)
        )
    ].copy()

    if len(entries) == 0:
        return None

    pitch, fig, ax = draw_pitch()

    passes_df = entries[entries["type"] == "Pass"]
    carries_df = entries[entries["type"] == "Carry"]

    if len(passes_df) > 0:
        pitch.arrows(
            passes_df["x"], passes_df["y"],
            passes_df["end_x"], passes_df["end_y"],
            ax=ax,
            width=2,
            headwidth=4,
            color=team_color,
            alpha=0.65,
            label="Pass"
        )

    if len(carries_df) > 0:
        pitch.arrows(
            carries_df["x"], carries_df["y"],
            carries_df["end_x"], carries_df["end_y"],
            ax=ax,
            width=2,
            headwidth=4,
            color="#111827",
            alpha=0.65,
            label="Carry"
        )

    ax.set_title(f"{team} | Box Entries", fontsize=15, weight="bold")
    ax.legend(loc="upper left")

    return fig


def plot_cross_origin_zones(match_events, match_id, team):
    df = match_events[
        (match_events["match_id"] == match_id) &
        (match_events["team"] == team) &
        (match_events["type"] == "Pass")
    ].copy()

    df = remove_penalty_shootout(df)

    if "pass_cross" not in df.columns:
        return None

    df = df[df["pass_cross"].fillna(False) == True].copy()
    df = ensure_xy(df)
    df = df.dropna(subset=["x", "y"])

    if len(df) == 0:
        return None

    pitch, fig, ax = draw_pitch()

    bin_statistic = pitch.bin_statistic(df["x"], df["y"], statistic="count", bins=(6, 4))
    pitch.heatmap(bin_statistic, ax=ax, cmap="Blues", alpha=0.78)
    pitch.label_heatmap(bin_statistic, color="#111827", fontsize=10, ax=ax)

    ax.set_title(f"{team} | Cross Origin Zones", fontsize=15, weight="bold")
    return fig


def plot_cross_target_zones(match_events, match_id, team):
    df = match_events[
        (match_events["match_id"] == match_id) &
        (match_events["team"] == team) &
        (match_events["type"] == "Pass")
    ].copy()

    df = remove_penalty_shootout(df)

    if "pass_cross" not in df.columns or "pass_end_location" not in df.columns:
        return None

    df = df[df["pass_cross"].fillna(False) == True].copy()

    pass_end = df["pass_end_location"].apply(parse_location)
    df["x"] = pass_end.apply(lambda v: v[0] if isinstance(v, list) and len(v) > 0 else np.nan)
    df["y"] = pass_end.apply(lambda v: v[1] if isinstance(v, list) and len(v) > 1 else np.nan)

    df = df.dropna(subset=["x", "y"])

    if len(df) == 0:
        return None

    pitch, fig, ax = draw_pitch()

    bin_statistic = pitch.bin_statistic(df["x"], df["y"], statistic="count", bins=(6, 4))
    pitch.heatmap(bin_statistic, ax=ax, cmap="Oranges", alpha=0.78)
    pitch.label_heatmap(bin_statistic, color="#111827", fontsize=10, ax=ax)

    ax.set_title(f"{team} | Cross Target Zones", fontsize=15, weight="bold")
    return fig


def plot_offensive_duel_zones(match_events, match_id, team):
    df = match_events[
        (match_events["match_id"] == match_id) &
        (match_events["team"] == team) &
        (match_events["type"] == "Duel")
    ].copy()

    df = remove_penalty_shootout(df)
    df = ensure_xy(df)
    df = df.dropna(subset=["x", "y"])
    df = df[df["x"] >= 60].copy()

    if len(df) == 0:
        return None

    pitch, fig, ax = draw_pitch()

    bin_statistic = pitch.bin_statistic(df["x"], df["y"], statistic="count", bins=(6, 4))
    pitch.heatmap(bin_statistic, ax=ax, cmap="Oranges", alpha=0.78)
    pitch.label_heatmap(bin_statistic, color="#111827", fontsize=10, ax=ax)

    ax.set_title(f"{team} | Offensive Duel Zones", fontsize=15, weight="bold")
    return fig


def plot_aerial_duel_zones(match_events, match_id, team):
    df = match_events[
        (match_events["match_id"] == match_id) &
        (match_events["team"] == team) &
        (match_events["type"] == "Duel")
    ].copy()

    df = remove_penalty_shootout(df)

    if "duel_type" in df.columns:
        df = df[df["duel_type"].astype(str).str.contains("Aerial", case=False, na=False)]

    df = ensure_xy(df)
    df = df.dropna(subset=["x", "y"])

    if len(df) == 0:
        return None

    pitch, fig, ax = draw_pitch()

    bin_statistic = pitch.bin_statistic(df["x"], df["y"], statistic="count", bins=(6, 4))
    pitch.heatmap(bin_statistic, ax=ax, cmap="Greens", alpha=0.78)
    pitch.label_heatmap(bin_statistic, color="#111827", fontsize=10, ax=ax)

    ax.set_title(f"{team} | Aerial Duel Zones", fontsize=15, weight="bold")
    return fig


def plot_vulnerable_zones(match_events, match_id, opponent, selected_team):
    df = match_events[
        (match_events["match_id"] == match_id) &
        (match_events["team"] == opponent) &
        (match_events["type"] == "Shot")
    ].copy()

    df = remove_penalty_shootout(df)
    df = ensure_xy(df)
    df = df.dropna(subset=["x", "y"])

    if len(df) == 0:
        return None

    pitch, fig, ax = draw_pitch()

    bin_statistic = pitch.bin_statistic(df["x"], df["y"], statistic="count", bins=(6, 4))
    pitch.heatmap(bin_statistic, ax=ax, cmap="Reds", alpha=0.78)
    pitch.label_heatmap(bin_statistic, color="#111827", fontsize=10, ax=ax)

    ax.set_title(f"{selected_team} | Vulnerable Zones - Opponent Shots", fontsize=15, weight="bold")
    return fig


def plot_pressure_height(team_events, team, team_color="#2563EB"):
    df = team_events[team_events["type"] == "Pressure"].copy()
    df = ensure_xy(df)
    df = df.dropna(subset=["x"])

    if len(df) == 0:
        return None

    low = len(df[df["x"] < 40])
    mid = len(df[(df["x"] >= 40) & (df["x"] < 80)])
    high = len(df[df["x"] >= 80])

    plot_df = pd.DataFrame({
        "Zone": ["Low Block", "Mid Block", "High Press"],
        "Pressures": [low, mid, high]
    })

    fig, ax = plt.subplots(figsize=(7, 4))

    ax.barh(plot_df["Zone"][::-1], plot_df["Pressures"][::-1], color=team_color, alpha=0.85)
    ax.set_title(f"{team} | Pressure Height", fontsize=15, weight="bold")
    ax.set_xlabel("Pressures")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    return fig


def plot_match_passing_network(match_events, match_id, team, team_color="#2563EB", min_passes=4):
    df = match_events[
        (match_events["match_id"] == match_id) &
        (match_events["team"] == team) &
        (match_events["type"] == "Pass")
    ].copy()

    df = remove_penalty_shootout(df)

    if "pass_recipient" not in df.columns and "pass_recipient_name" in df.columns:
        df["pass_recipient"] = df["pass_recipient_name"]

    required_cols = ["player", "pass_recipient", "location", "pass_end_location"]

    if len(df) == 0 or not all(col in df.columns for col in required_cols):
        return None, pd.DataFrame()

    df = ensure_xy(df)

    pass_end = df["pass_end_location"].apply(parse_location)
    df["end_x"] = pass_end.apply(lambda v: v[0] if isinstance(v, list) and len(v) > 0 else np.nan)
    df["end_y"] = pass_end.apply(lambda v: v[1] if isinstance(v, list) and len(v) > 1 else np.nan)

    df = df.dropna(subset=["player", "pass_recipient", "x", "y", "end_x", "end_y"])

    if len(df) == 0:
        return None, pd.DataFrame()

    lineup = get_lineup_df(match_events, match_id, team)
    starters = set(lineup["player"].tolist()) if len(lineup) > 0 else set()

    avg_positions = (
        df.groupby("player")
        .agg(
            x=("x", "mean"),
            y=("y", "mean"),
            passes=("type", "count")
        )
        .reset_index()
    )

    avg_positions = avg_positions.merge(
        lineup[["player", "position", "position_order"]],
        on="player",
        how="left"
    )

    avg_positions["status"] = avg_positions["player"].apply(
        lambda x: "Starter" if x in starters else "Substitute"
    )

    avg_positions["position"] = avg_positions["position"].fillna("Substitute")
    avg_positions["position_order"] = avg_positions["position_order"].fillna(99)

    starters_df_order = (
        avg_positions[avg_positions["status"] == "Starter"]
        .sort_values(["position_order", "y"], ascending=[True, True])
        .copy()
    )

    subs_df_order = (
        avg_positions[avg_positions["status"] == "Substitute"]
        .sort_values("passes", ascending=False)
        .copy()
    )

    avg_positions = pd.concat([starters_df_order, subs_df_order], ignore_index=True)
    avg_positions["number"] = range(1, len(avg_positions) + 1)

    pass_links = (
        df.groupby(["player", "pass_recipient"])
        .size()
        .reset_index(name="passes")
    )

    pass_links = pass_links[pass_links["passes"] >= min_passes]

    legend_df = avg_positions[["number", "player", "position", "status", "passes"]].copy()
    legend_df.columns = ["#", "Player", "Position", "Status", "Passes"]

    if len(pass_links) == 0:
        return None, legend_df

    pitch, fig, ax = draw_pitch()

    for _, row in pass_links.iterrows():
        passer = avg_positions[avg_positions["player"] == row["player"]]
        receiver = avg_positions[avg_positions["player"] == row["pass_recipient"]]

        if len(passer) == 0 or len(receiver) == 0:
            continue

        x1 = passer.iloc[0]["x"]
        y1 = passer.iloc[0]["y"]
        x2 = receiver.iloc[0]["x"]
        y2 = receiver.iloc[0]["y"]

        ax.plot(
            [x1, x2],
            [y1, y2],
            linewidth=max(1, row["passes"] * 0.18),
            color=team_color,
            alpha=0.35,
            zorder=2
        )

    starters_plot = avg_positions[avg_positions["status"] == "Starter"]
    subs_plot = avg_positions[avg_positions["status"] == "Substitute"]

    if len(starters_plot) > 0:
        pitch.scatter(
            starters_plot["x"],
            starters_plot["y"],
            s=starters_plot["passes"] * 14 + 160,
            ax=ax,
            color=team_color,
            edgecolors="#111827",
            linewidth=1.3,
            alpha=0.95,
            zorder=3,
            label="Starter"
        )

    if len(subs_plot) > 0:
        pitch.scatter(
            subs_plot["x"],
            subs_plot["y"],
            s=subs_plot["passes"] * 14 + 160,
            ax=ax,
            color="#6B7280",
            edgecolors="#111827",
            linewidth=1.3,
            alpha=0.95,
            zorder=3,
            label="Substitute"
        )

    for _, row in avg_positions.iterrows():
        ax.text(
            row["x"],
            row["y"],
            str(row["number"]),
            ha="center",
            va="center",
            fontsize=10,
            color="white",
            weight="bold",
            zorder=4
        )

    ax.set_title(f"{team} | Passing Network", fontsize=15, weight="bold")
    ax.legend(loc="upper left")

    return fig, legend_df


def plot_barh(df, metric, title, color="#2563EB", top_n=8):
    if len(df) == 0 or metric not in df.columns:
        return None

    plot_df = df.sort_values(metric, ascending=False).head(top_n).copy()

    if len(plot_df) == 0:
        return None

    fig, ax = plt.subplots(figsize=(8, 4))

    ax.barh(plot_df["player"][::-1], plot_df[metric][::-1], color=color, alpha=0.85)
    ax.set_title(title, fontsize=15, weight="bold")
    ax.set_xlabel(metric)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    return fig


def plot_event_volume_comparison(team_events, opponent_events, team, opponent, team_color, opponent_color):
    phases = {
        "Shots": "Shot",
        "Passes": "Pass",
        "Carries": "Carry",
        "Pressures": "Pressure",
        "Recoveries": ["Ball Recovery", "Interception"],
        "Fouls": "Foul Committed"
    }

    rows = []

    for label, event_type in phases.items():
        if isinstance(event_type, list):
            team_value = team_events["event_type"].isin(event_type).sum()
            opponent_value = opponent_events["event_type"].isin(event_type).sum()
        else:
            team_value = (team_events["event_type"] == event_type).sum()
            opponent_value = (opponent_events["event_type"] == event_type).sum()

        rows.append({
            "phase": label,
            team: team_value,
            opponent: opponent_value
        })

    plot_df = pd.DataFrame(rows)

    fig, ax = plt.subplots(figsize=(8, 4))

    x = np.arange(len(plot_df))
    width = 0.36

    ax.bar(x - width / 2, plot_df[team], width, label=team, color=team_color, alpha=0.85)
    ax.bar(x + width / 2, plot_df[opponent], width, label=opponent, color=opponent_color, alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(plot_df["phase"], rotation=0)
    ax.set_ylabel("Events")
    ax.set_title("Event Volume Comparison", fontsize=15, weight="bold")
    ax.legend(loc="upper left")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    return fig


st.sidebar.title("Filters")

tournament_id = st.sidebar.selectbox(
    "Tournament",
    sorted(matches["tournament_id"].dropna().unique())
)

tournament_matches = matches[matches["tournament_id"] == tournament_id].copy()

if "match_label" not in tournament_matches.columns:
    tournament_matches["match_label"] = build_match_label(tournament_matches)

match_label = st.sidebar.selectbox(
    "Match",
    tournament_matches["match_label"].tolist()
)

match_row = tournament_matches[tournament_matches["match_label"] == match_label].iloc[0]

match_id = match_row["match_id"]

home_team = match_row["home_team"]
away_team = match_row["away_team"]

team = st.sidebar.selectbox(
    "Team",
    [home_team, away_team]
)

opponent = away_team if team == home_team else home_team

team_flag, team_color = get_team_asset(team)
opponent_flag, opponent_color = get_team_asset(opponent)
home_flag, home_color = get_team_asset(home_team)
away_flag, away_color = get_team_asset(away_team)

match_events = events[events["match_id"] == match_id].copy()
match_play_events = remove_penalty_shootout(match_events)

match_events = ensure_xy(match_events)
match_play_events = ensure_xy(match_play_events)

team_events = match_play_events[match_play_events["team"] == team].copy()
opponent_events = match_play_events[match_play_events["team"] == opponent].copy()

shots = len(team_events[team_events["event_type"] == "Shot"])
opponent_shots = len(opponent_events[opponent_events["event_type"] == "Shot"])

passes = len(team_events[team_events["event_type"] == "Pass"])
pressures = len(team_events[team_events["event_type"] == "Pressure"])
recoveries = len(team_events[team_events["event_type"].isin(["Ball Recovery", "Interception"])])
carries = len(team_events[team_events["event_type"] == "Carry"])
duels = len(team_events[team_events["event_type"] == "Duel"])

xg = team_events["shot_statsbomb_xg"].fillna(0).sum() if "shot_statsbomb_xg" in team_events.columns else 0
opponent_xg = opponent_events["shot_statsbomb_xg"].fillna(0).sum() if "shot_statsbomb_xg" in opponent_events.columns else 0

if team == home_team:
    goals_for = match_row["home_score"]
    goals_against = match_row["away_score"]
else:
    goals_for = match_row["away_score"]
    goals_against = match_row["home_score"]

result = "Draw"

if goals_for > goals_against:
    result = "Win"
elif goals_for < goals_against:
    result = "Loss"

player_match = build_player_match_table(team_events)
player_match = add_progressive_metrics_to_players(team_events, player_match)

competition_name = get_row_value(match_row, ["competition_name", "competition"], "Unknown Competition")
season_name = get_row_value(match_row, ["season_name", "season"], "Unknown Season")
stage_name = get_row_value(match_row, ["competition_stage", "stage_name"], "")


header_left, header_center, header_right = st.columns([1, 6, 1])

with header_left:
    if home_flag:
        st.image(home_flag, width=90)

with header_center:
    st.markdown(
        f"""
        <h1 style='text-align:center; margin-bottom:0;'>
            {home_team} {match_row['home_score']} - {match_row['away_score']} {away_team}
        </h1>
        """,
        unsafe_allow_html=True
    )

    st.markdown(
        f"<div class='page-subtitle' style='text-align:center;'>{competition_name} | {season_name} | {stage_name}</div>",
        unsafe_allow_html=True
    )

    st.markdown(
        f"<div class='page-subtitle' style='text-align:center;'>Selected team: <b>{team}</b> vs {opponent}</div>",
        unsafe_allow_html=True
    )

with header_right:
    if away_flag:
        st.image(away_flag, width=90)


c1, c2, c3, c4, c5, c6 = st.columns(6)

with c1:
    metric_card("Result", result)

with c2:
    metric_card("Goals", int(goals_for))

with c3:
    metric_card("xG", round(xg, 2))

with c4:
    metric_card("Shots", shots)

with c5:
    metric_card("Passes", passes)

with c6:
    metric_card("Pressures", pressures)


tab_overview, tab_build, tab_attack, tab_defense, tab_players = st.tabs([
    "Overview",
    "Build-Up",
    "Attack",
    "Defense",
    "Players"
])


with tab_overview:

    st.markdown("## Match Overview")

    summary = (
        f"{team} finished the match with {shots} shots and {round(xg, 2)} xG, "
        f"compared with {opponent}'s {opponent_shots} shots and {round(opponent_xg, 2)} xG. "
        f"The selected team recorded {passes} passes, {pressures} pressures, "
        f"{recoveries} recoveries/interceptions and {duels} duels."
    )

    text_card("Match Summary", summary)

    st.markdown("### Starting XI")

    fig = plot_match_xi_comparison(
        match_events,
        match_id,
        home_team,
        away_team,
        home_color=home_color,
        away_color=away_color
    )

    if fig is not None:
        st.pyplot(fig, use_container_width=True)
    else:
        st.info("No Starting XI data available.")

    st.markdown("### Match Control")

    m1, m2 = st.columns(2)

    with m1:
        fig = plot_match_xg_flow(
            match_events,
            match_id,
            team,
            opponent,
            team_color=team_color,
            opponent_color=opponent_color
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No xG flow available.")

    with m2:
        fig = plot_event_volume_comparison(
            team_events,
            opponent_events,
            team,
            opponent,
            team_color,
            opponent_color
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)

    st.markdown("### Key Players")

    fig = plot_barh(
        player_match,
        "impact_score",
        f"{team} | Key Players Impact",
        color=team_color
    )

    if fig is not None:
        st.pyplot(fig, use_container_width=True)


with tab_build:

    st.markdown("## Build-Up & Control")

    b1, b2, b3, b4 = st.columns(4)

    with b1:
        metric_card("Passes", passes)

    with b2:
        metric_card("Carries", carries)

    with b3:
        metric_card("Pass / Carry Ratio", round(passes / carries, 2) if carries > 0 else 0)

    with b4:
        metric_card("Total On-Ball Events", passes + carries)

    st.markdown("### Passing Network")

    network_col, legend_col = st.columns([2.2, 1])

    with network_col:
        fig, network_legend = plot_match_passing_network(
            match_events,
            match_id,
            team,
            team_color=team_color,
            min_passes=4
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No passing network available.")

    with legend_col:
        st.markdown("#### Player Legend")

        if len(network_legend) > 0:
            st.dataframe(
                network_legend,
                use_container_width=True,
                hide_index=True,
                height=520
            )
        else:
            st.info("No legend available.")

    st.markdown("### Zones")

    z1, z2 = st.columns(2)

    with z1:
        fig = plot_match_zone_heatmap(
            match_events,
            match_id,
            team,
            ["Pass"],
            f"{team} | Pass Origin Zones",
            cmap="Blues"
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)

    with z2:
        fig = plot_match_zone_heatmap(
            match_events,
            match_id,
            team,
            ["Carry"],
            f"{team} | Carry Zones",
            cmap="Purples"
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)


with tab_attack:

    st.markdown("## Offensive Analysis")

    o1, o2, o3, o4 = st.columns(4)

    with o1:
        metric_card("Shots", shots)

    with o2:
        metric_card("xG", round(xg, 2))

    with o3:
        metric_card("Goals", int(goals_for))

    with o4:
        conversion = round((goals_for / shots) * 100, 1) if shots > 0 else 0
        metric_card("Shot Conversion", f"{conversion}%")

    s1, s2 = st.columns(2)

    with s1:
        fig = plot_match_shot_map(
            match_events,
            match_id,
            team,
            team_color=team_color
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No shot map available.")

    with s2:
        fig = plot_box_entries(
            match_events,
            match_id,
            team,
            team_color=team_color
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No box entries available.")

    st.markdown("### Crosses & Offensive Duels")

    a1, a2 = st.columns(2)

    with a1:
        fig = plot_cross_origin_zones(
            match_events,
            match_id,
            team
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No cross origin data available.")

    with a2:
        fig = plot_cross_target_zones(
            match_events,
            match_id,
            team
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No cross target data available.")

    st.markdown("### Offensive Duels")

    fig = plot_offensive_duel_zones(
        match_events,
        match_id,
        team
    )

    if fig is not None:
        st.pyplot(fig, use_container_width=True)
    else:
        st.info("No offensive duel data available.")


with tab_defense:

    st.markdown("## Defensive Analysis")

    d1, d2, d3, d4 = st.columns(4)

    with d1:
        metric_card("Pressures", pressures)

    with d2:
        metric_card("Recoveries + Interceptions", recoveries)

    with d3:
        metric_card("Duels", duels)

    with d4:
        metric_card("Opponent Shots", opponent_shots)

    r1, r2 = st.columns(2)

    with r1:
        fig = plot_match_zone_heatmap(
            match_events,
            match_id,
            team,
            ["Ball Recovery", "Interception"],
            f"{team} | Recovery Zones",
            cmap="Greens"
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)

    with r2:
        fig = plot_pressure_height(
            team_events,
            team,
            team_color=team_color
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)

    st.markdown("### Defensive Risk")

    dcol1, dcol2 = st.columns(2)

    with dcol1:
        fig = plot_vulnerable_zones(
            match_events,
            match_id,
            opponent,
            team
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No vulnerable zone data available.")

    with dcol2:
        fig = plot_aerial_duel_zones(
            match_events,
            match_id,
            team
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No aerial duel data available.")


with tab_players:

    st.markdown("## Player Match Impact")

    p1, p2 = st.columns(2)

    with p1:
        fig = plot_barh(
            player_match,
            "impact_score",
            f"{team} | Key Players Impact",
            color=team_color
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)

    with p2:
        fig = plot_barh(
            player_match,
            "events",
            f"{team} | Most Involved Players",
            color=team_color
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)

    p3, p4 = st.columns(2)

    with p3:
        fig = plot_barh(
            player_match,
            "pressures",
            f"{team} | Most Active Pressers",
            color=team_color
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)

    with p4:
        fig = plot_barh(
            player_match,
            "progressive_actions",
            f"{team} | Progressive Players",
            color=team_color
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)

    st.markdown("### Full Player Match Table")

    table_cols = [
        "player",
        "impact_score",
        "events",
        "shots",
        "xg",
        "passes",
        "carries",
        "progressive_actions",
        "pressures",
        "recoveries",
        "duels",
        "fouls"
    ]

    existing_cols = [col for col in table_cols if col in player_match.columns]

    st.dataframe(
        player_match[existing_cols],
        use_container_width=True,
        hide_index=True
    )


st.divider()
st.caption(
    "Developed by Juan Ignacio Breccia | Football Analytics · Scouting · Data Science | Powered by StatsBomb Open Data"
)
'''

with open(pages_path / "2_Team_Match.py", "w", encoding="utf-8") as f:
    f.write(team_match_code.strip())

print("2_Team_Match.py updated successfully")

2_Team_Match.py updated successfully


In [7]:
from pathlib import Path

pages_path = Path("streamlit_app/pages")
pages_path.mkdir(parents=True, exist_ok=True)

team_comparison_code = r'''
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import streamlit as st
from pathlib import Path

from utils.data_loader import load_all_data
from utils.filters import get_tournament_options, get_team_options, filter_team
from utils.style import apply_global_style
from utils.cards import metric_card, text_card


st.set_page_config(page_title="Team Comparison", layout="wide")
apply_global_style()

data = load_all_data()

matches = data["matches"]

if "team_master" not in data:
    st.error("team_master not found. Run team aggregation notebook first.")
    st.stop()

team_master = data["team_master"]
assets = data.get("assets", pd.DataFrame())

BASE_DIR = Path(__file__).resolve().parents[2]


# ============================================================
# HELPERS
# ============================================================

def get_team_asset(team_name):

    if assets is None or len(assets) == 0:
        return None, "#2563EB"

    if "team" not in assets.columns:
        return None, "#2563EB"

    asset_row = assets[assets["team"] == team_name]

    if len(asset_row) == 0:
        return None, "#2563EB"

    primary_color = asset_row.iloc[0].get("primary_color", "#2563EB")
    raw_flag_path = asset_row.iloc[0].get("flag_path", None)

    if pd.isna(raw_flag_path) or raw_flag_path is None:
        return None, primary_color

    raw_flag_path = str(raw_flag_path)

    possible_paths = [
        BASE_DIR / raw_flag_path,
        BASE_DIR / "streamlit_app" / raw_flag_path,
        Path(raw_flag_path),
    ]

    for p in possible_paths:
        if p.exists():
            return str(p), primary_color

    return None, primary_color


def get_row(profile):
    if len(profile) == 0:
        return None
    return profile.iloc[0]


def get_value(row, col, default=0):

    if row is None:
        return default

    if col not in row.index:
        return default

    value = row[col]

    if pd.isna(value):
        return default

    return value


def safe_metric(row, col, fallback=None):
    if fallback is None:
        return get_value(row, col, 0)

    return get_value(row, col, get_value(row, fallback, 0))


def render_team_header(team_name, tournament_name, flag_path, color):
    c1, c2 = st.columns([1, 4])

    with c1:
        if flag_path:
            st.image(flag_path, width=70)

    with c2:
        st.markdown(
            f"<h2 style='color:{color}; margin-bottom:0;'>{team_name}</h2>",
            unsafe_allow_html=True
        )
        st.markdown(
            f"<div class='page-subtitle'>{tournament_name}</div>",
            unsafe_allow_html=True
        )


def render_team_side(team_name, tournament_name, row, flag_path, color):
    render_team_header(team_name, tournament_name, flag_path, color)

    st.markdown("")

    c1, c2 = st.columns(2)

    with c1:
        metric_card("Matches", int(get_value(row, "matches", 0) or 0))

    with c2:
        metric_card("Goals", int(safe_metric(row, "goals_for", "goals") or 0))

    c3, c4 = st.columns(2)

    with c3:
        metric_card("Attack", round(get_value(row, "attack_score", 0) or 0, 1))

    with c4:
        metric_card("Defense", round(get_value(row, "defense_score", 0) or 0, 1))

    st.markdown("#### Style")

    text_card(
        "Tactical Identity",
        (
            f"General: {get_value(row, 'general_style', 'N/A')}<br>"
            f"Offensive: {get_value(row, 'offensive_style', 'N/A')}<br>"
            f"Defensive: {get_value(row, 'defensive_style', 'N/A')}"
        )
    )

    st.markdown("#### Key Metrics")

    metrics_df = pd.DataFrame([
        {"Metric": "Build-Up Score", "Value": round(get_value(row, "build_up_score", 0) or 0, 1)},
        {"Metric": "Directness Score", "Value": round(get_value(row, "directness_score", 0) or 0, 1)},
        {"Metric": "Passes / Match", "Value": round(get_value(row, "passes_per_match", 0) or 0, 1)},
        {"Metric": "Pass Accuracy %", "Value": round(get_value(row, "avg_pass_completion_pct", 0) or 0, 1)},
        {"Metric": "Tempo", "Value": round(get_value(row, "tempo", 0) or 0, 2)},
        {"Metric": "Pressures / Match", "Value": round(get_value(row, "pressures_per_match", 0) or 0, 1)},
        {"Metric": "Recoveries / Match", "Value": round(get_value(row, "recoveries_per_match", 0) or 0, 1)},
    ])

    st.dataframe(
        metrics_df,
        use_container_width=True,
        hide_index=True,
        height=285
    )


def plot_two_team_radar(row_a, row_b, team_a, team_b, color_a, color_b):
    metrics = [
        ("Attack", "attack_score"),
        ("Build-Up", "build_up_score"),
        ("Defense", "defense_score"),
        ("Directness", "directness_score"),
    ]

    labels = [m[0] for m in metrics]

    values_a = [float(get_value(row_a, col, 0) or 0) for _, col in metrics]
    values_b = [float(get_value(row_b, col, 0) or 0) for _, col in metrics]

    values_a += values_a[:1]
    values_b += values_b[:1]

    angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(7.5, 7.5), subplot_kw=dict(polar=True))

    ax.plot(angles, values_a, linewidth=2.5, color=color_a, label=team_a)
    ax.fill(angles, values_a, color=color_a, alpha=0.16)

    ax.plot(angles, values_b, linewidth=2.5, color=color_b, label=team_b)
    ax.fill(angles, values_b, color=color_b, alpha=0.16)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=11, weight="bold")

    max_val = max(
        max(values_a),
        max(values_b),
        1
    )

    upper = np.ceil(max_val / 10) * 10
    ax.set_ylim(0, upper)

    ax.set_title(
        f"{team_a} vs {team_b}",
        fontsize=18,
        weight="bold",
        pad=28
    )

    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=2)

    return fig


# ============================================================
# SIDEBAR
# ============================================================

st.sidebar.title("Filters")

tournaments = get_tournament_options(matches)

tournament_a = st.sidebar.selectbox(
    "Tournament A",
    tournaments,
    index=0
)

teams_a = get_team_options(team_master, tournament_a)

team_a = st.sidebar.selectbox(
    "Team A",
    teams_a,
    index=0
)

default_idx = min(1, len(tournaments) - 1)

tournament_b = st.sidebar.selectbox(
    "Tournament B",
    tournaments,
    index=default_idx
)

teams_b = get_team_options(team_master, tournament_b)

team_b = st.sidebar.selectbox(
    "Team B",
    teams_b,
    index=0
)

team_a_profile = filter_team(team_master, tournament_a, team_a)
team_b_profile = filter_team(team_master, tournament_b, team_b)

row_a = get_row(team_a_profile)
row_b = get_row(team_b_profile)

if row_a is None:
    st.error(f"No data found for {team_a}")
    st.stop()

if row_b is None:
    st.error(f"No data found for {team_b}")
    st.stop()

team_a_flag, team_a_color = get_team_asset(team_a)
team_b_flag, team_b_color = get_team_asset(team_b)


# ============================================================
# HEADER
# ============================================================

h1, h2, h3 = st.columns([1, 6, 1])

with h1:
    if team_a_flag:
        st.image(team_a_flag, width=85)

with h2:
    st.markdown(
        f"<h1 style='text-align:center; margin-bottom:0;'>{team_a} vs {team_b}</h1>",
        unsafe_allow_html=True
    )
    st.markdown(
        f"<div class='page-subtitle' style='text-align:center;'>{team_a} ({tournament_a}) vs {team_b} ({tournament_b})</div>",
        unsafe_allow_html=True
    )

with h3:
    if team_b_flag:
        st.image(team_b_flag, width=85)


# ============================================================
# KPI ROW
# ============================================================

k1, k2, k3, k4, k5, k6 = st.columns(6)

with k1:
    metric_card(f"{team_a} Goals", int(safe_metric(row_a, "goals_for", "goals") or 0))

with k2:
    metric_card(f"{team_b} Goals", int(safe_metric(row_b, "goals_for", "goals") or 0))

with k3:
    metric_card(f"{team_a} Attack", round(get_value(row_a, "attack_score", 0) or 0, 1))

with k4:
    metric_card(f"{team_b} Attack", round(get_value(row_b, "attack_score", 0) or 0, 1))

with k5:
    metric_card(f"{team_a} Defense", round(get_value(row_a, "defense_score", 0) or 0, 1))

with k6:
    metric_card(f"{team_b} Defense", round(get_value(row_b, "defense_score", 0) or 0, 1))


# ============================================================
# MAIN VIEW
# ============================================================

left_col, radar_col, right_col = st.columns([1.15, 1.45, 1.15])

with left_col:
    render_team_side(
        team_a,
        tournament_a,
        row_a,
        team_a_flag,
        team_a_color
    )

with radar_col:
    st.markdown("### Team DNA Radar")

    fig = plot_two_team_radar(
        row_a,
        row_b,
        team_a,
        team_b,
        team_a_color,
        team_b_color
    )

    if fig is not None:
        st.pyplot(fig, use_container_width=True)

    st.markdown("### Quick Comparison")

    quick_df = pd.DataFrame([
        {
            "Metric": "Attack Score",
            team_a: round(get_value(row_a, "attack_score", 0) or 0, 1),
            team_b: round(get_value(row_b, "attack_score", 0) or 0, 1),
        },
        {
            "Metric": "Build-Up Score",
            team_a: round(get_value(row_a, "build_up_score", 0) or 0, 1),
            team_b: round(get_value(row_b, "build_up_score", 0) or 0, 1),
        },
        {
            "Metric": "Defense Score",
            team_a: round(get_value(row_a, "defense_score", 0) or 0, 1),
            team_b: round(get_value(row_b, "defense_score", 0) or 0, 1),
        },
        {
            "Metric": "Directness Score",
            team_a: round(get_value(row_a, "directness_score", 0) or 0, 1),
            team_b: round(get_value(row_b, "directness_score", 0) or 0, 1),
        },
        {
            "Metric": "Passes / Match",
            team_a: round(get_value(row_a, "passes_per_match", 0) or 0, 1),
            team_b: round(get_value(row_b, "passes_per_match", 0) or 0, 1),
        },
        {
            "Metric": "Pressures / Match",
            team_a: round(get_value(row_a, "pressures_per_match", 0) or 0, 1),
            team_b: round(get_value(row_b, "pressures_per_match", 0) or 0, 1),
        },
    ])

    st.dataframe(
        quick_df,
        use_container_width=True,
        hide_index=True,
        height=280
    )

with right_col:
    render_team_side(
        team_b,
        tournament_b,
        row_b,
        team_b_flag,
        team_b_color
    )


# ============================================================
# FOOTER
# ============================================================

st.divider()

st.caption(
    "Developed by Juan Ignacio Breccia | Football Analytics · Scouting · Data Science | Powered by StatsBomb Open Data"
)
'''

with open(pages_path / "3_Team_Comparison.py", "w", encoding="utf-8") as f:
    f.write(team_comparison_code.strip())

print("3_Team_Comparison.py updated successfully")

3_Team_Comparison.py updated successfully


In [437]:
from pathlib import Path

pages_path = Path("streamlit_app/pages")
pages_path.mkdir(parents=True, exist_ok=True)

player_match_code = r'''
import ast
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import streamlit as st
from mplsoccer import Pitch

from utils.data_loader import load_all_data
from utils.filters import get_tournament_options, get_team_options, get_player_options
from utils.style import apply_global_style
from utils.cards import metric_card, text_card


st.set_page_config(page_title="Player Match", layout="wide")
apply_global_style()

data = load_all_data()

matches = data["matches"]
events = data["events"]
assets = data["assets"]
player_match_stats = data.get("player_match_stats_with_positions", None)

if "event_type" not in events.columns and "type" in events.columns:
    events["event_type"] = events["type"]

BASE_DIR = Path(__file__).resolve().parents[2]


# ============================================================
# HELPERS
# ============================================================

def parse_location(value):
    if isinstance(value, list):
        return value

    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            return None

    return None


def ensure_xy(df):
    df = df.copy()

    if "x" not in df.columns:
        df["x"] = np.nan

    if "y" not in df.columns:
        df["y"] = np.nan

    if "location" in df.columns:
        loc = df["location"].apply(parse_location)

        df["x"] = df["x"].fillna(
            loc.apply(lambda v: v[0] if isinstance(v, list) and len(v) > 0 else np.nan)
        )

        df["y"] = df["y"].fillna(
            loc.apply(lambda v: v[1] if isinstance(v, list) and len(v) > 1 else np.nan)
        )

    return df


def remove_penalty_shootout(df):
    df = df.copy()

    if "period" in df.columns:
        df = df[df["period"].fillna(0).astype(float) <= 4].copy()

    return df


def draw_pitch():
    pitch = Pitch(
        pitch_type="statsbomb",
        pitch_color="#F8FAFC",
        line_color="#111827",
        linewidth=1.1
    )

    fig, ax = pitch.draw(figsize=(10, 7))
    return pitch, fig, ax


def get_team_asset(team_name):
    asset_row = assets[assets["team"] == team_name]

    if len(asset_row) == 0:
        return None, "#2563EB"

    primary_color = asset_row.iloc[0].get("primary_color", "#2563EB")
    raw_flag_path = asset_row.iloc[0].get("flag_path", None)

    if not raw_flag_path or pd.isna(raw_flag_path):
        return None, primary_color

    raw_flag_path = str(raw_flag_path)

    possible_paths = [
        BASE_DIR / raw_flag_path,
        BASE_DIR / "streamlit_app" / raw_flag_path,
        Path(raw_flag_path)
    ]

    for p in possible_paths:
        if p.exists():
            return str(p), primary_color

    return None, primary_color


def build_match_label(df):
    return (
        df["match_date"].astype(str)
        + " | "
        + df["home_team"].astype(str)
        + " "
        + df["home_score"].astype(str)
        + "-"
        + df["away_score"].astype(str)
        + " "
        + df["away_team"].astype(str)
    )


def pct(numerator, denominator):
    if denominator == 0:
        return 0
    return round((numerator / denominator) * 100, 1)


def build_kpis(player_events):
    shots = len(player_events[player_events["event_type"] == "Shot"])
    passes = len(player_events[player_events["event_type"] == "Pass"])
    carries = len(player_events[player_events["event_type"] == "Carry"])
    pressures = len(player_events[player_events["event_type"] == "Pressure"])
    recoveries = len(player_events[player_events["event_type"].isin(["Ball Recovery", "Interception"])])
    duels = len(player_events[player_events["event_type"] == "Duel"])
    fouls = len(player_events[player_events["event_type"] == "Foul Committed"])
    xg = player_events["shot_statsbomb_xg"].fillna(0).sum() if "shot_statsbomb_xg" in player_events.columns else 0

    goals = 0
    shots_on_target = 0

    if "shot_outcome" in player_events.columns:
        goals = len(player_events[player_events["shot_outcome"] == "Goal"])
        shots_on_target = len(
            player_events[
                (player_events["event_type"] == "Shot") &
                (player_events["shot_outcome"].astype(str).isin(["Goal", "Saved", "Saved To Post"]))
            ]
        )

    completed_passes = 0

    if "pass_outcome" in player_events.columns:
        completed_passes = len(
            player_events[
                (player_events["event_type"] == "Pass") &
                (player_events["pass_outcome"].isna())
            ]
        )

    crosses = 0
    completed_crosses = 0

    if "pass_cross" in player_events.columns:
        cross_df = player_events[
            (player_events["event_type"] == "Pass") &
            (player_events["pass_cross"].fillna(False) == True)
        ].copy()

        crosses = len(cross_df)

        if "pass_outcome" in cross_df.columns:
            completed_crosses = len(cross_df[cross_df["pass_outcome"].isna()])

    duels_won = 0

    if "duel_outcome" in player_events.columns:
        duels_won = len(
            player_events[
                (player_events["event_type"] == "Duel") &
                (player_events["duel_outcome"].astype(str).str.contains("Won|Success", case=False, na=False))
            ]
        )

    return {
        "events": len(player_events),
        "shots": shots,
        "goals": goals,
        "xg": xg,
        "shots_on_target": shots_on_target,
        "shot_accuracy_pct": pct(shots_on_target, shots),
        "conversion_pct": pct(goals, shots),
        "passes": passes,
        "completed_passes": completed_passes,
        "pass_accuracy_pct": pct(completed_passes, passes),
        "carries": carries,
        "pressures": pressures,
        "recoveries": recoveries,
        "duels": duels,
        "duels_won": duels_won,
        "duel_win_pct": pct(duels_won, duels),
        "fouls": fouls,
        "crosses": crosses,
        "completed_crosses": completed_crosses,
        "cross_accuracy_pct": pct(completed_crosses, crosses)
    }


# ============================================================
# VISUALS
# ============================================================

def plot_action_map(player_events, player, team_color="#2563EB"):
    df = ensure_xy(player_events)
    df = df.dropna(subset=["x", "y"])

    if len(df) == 0:
        return None

    pitch, fig, ax = draw_pitch()

    event_colors = {
        "Shot": "#DC2626",
        "Pass": team_color,
        "Carry": "#7C3AED",
        "Pressure": "#F97316",
        "Ball Recovery": "#16A34A",
        "Interception": "#15803D",
        "Duel": "#6B7280"
    }

    for event_type, color in event_colors.items():
        plot_df = df[df["event_type"] == event_type]

        if len(plot_df) == 0:
            continue

        pitch.scatter(
            plot_df["x"],
            plot_df["y"],
            s=80,
            ax=ax,
            color=color,
            alpha=0.65,
            label=event_type
        )

    ax.set_title(f"{player} | Match Action Map", fontsize=15, weight="bold")
    ax.legend(loc="upper left")

    return fig


def plot_zone_heatmap(player_events, event_types, title, cmap="Blues"):
    df = player_events[player_events["event_type"].isin(event_types)].copy()

    df = ensure_xy(df)
    df = df.dropna(subset=["x", "y"])

    if len(df) == 0:
        return None

    pitch, fig, ax = draw_pitch()

    bin_statistic = pitch.bin_statistic(
        df["x"],
        df["y"],
        statistic="count",
        bins=(6, 4)
    )

    pitch.heatmap(bin_statistic, ax=ax, cmap=cmap, alpha=0.78)
    pitch.label_heatmap(
        bin_statistic,
        color="#111827",
        fontsize=10,
        ax=ax,
        ha="center",
        va="center"
    )

    ax.set_title(title, fontsize=15, weight="bold")

    return fig


def plot_shot_map(player_events, player, team_color="#2563EB"):
    df = player_events[player_events["event_type"] == "Shot"].copy()

    df = ensure_xy(df)
    df = df.dropna(subset=["x", "y"])

    if len(df) == 0:
        return None

    if "shot_statsbomb_xg" not in df.columns:
        df["shot_statsbomb_xg"] = 0

    df["shot_xg"] = df["shot_statsbomb_xg"].fillna(0)

    pitch, fig, ax = draw_pitch()

    goals = df[df["shot_outcome"] == "Goal"] if "shot_outcome" in df.columns else pd.DataFrame()
    shots = df[df["shot_outcome"] != "Goal"] if "shot_outcome" in df.columns else df

    if len(shots) > 0:
        pitch.scatter(
            shots["x"],
            shots["y"],
            s=shots["shot_xg"] * 1300 + 65,
            ax=ax,
            color=team_color,
            alpha=0.5,
            label="Shot"
        )

    if len(goals) > 0:
        pitch.scatter(
            goals["x"],
            goals["y"],
            s=goals["shot_xg"] * 1500 + 100,
            ax=ax,
            color="#DC2626",
            edgecolors="#111827",
            linewidth=1.5,
            label="Goal"
        )

    ax.set_title(
        f"{player} Shot Map | Shots {len(df)} | Goals {len(goals)} | xG {df['shot_xg'].sum():.2f}",
        fontsize=15,
        weight="bold"
    )

    ax.legend(loc="upper left")

    return fig


def plot_cross_origin_zones(player_events, player):
    if "pass_cross" not in player_events.columns:
        return None

    df = player_events[
        (player_events["event_type"] == "Pass") &
        (player_events["pass_cross"].fillna(False) == True)
    ].copy()

    return plot_zone_heatmap(
        df,
        ["Pass"],
        f"{player} | Cross Origin Zones",
        cmap="Blues"
    )


def plot_cross_target_zones(player_events, player):
    if "pass_cross" not in player_events.columns or "pass_end_location" not in player_events.columns:
        return None

    df = player_events[
        (player_events["event_type"] == "Pass") &
        (player_events["pass_cross"].fillna(False) == True)
    ].copy()

    if len(df) == 0:
        return None

    pass_end = df["pass_end_location"].apply(parse_location)

    df["x"] = pass_end.apply(lambda v: v[0] if isinstance(v, list) and len(v) > 0 else np.nan)
    df["y"] = pass_end.apply(lambda v: v[1] if isinstance(v, list) and len(v) > 1 else np.nan)

    df = df.dropna(subset=["x", "y"])

    if len(df) == 0:
        return None

    pitch, fig, ax = draw_pitch()

    bin_statistic = pitch.bin_statistic(df["x"], df["y"], statistic="count", bins=(6, 4))
    pitch.heatmap(bin_statistic, ax=ax, cmap="Oranges", alpha=0.78)
    pitch.label_heatmap(bin_statistic, color="#111827", fontsize=10, ax=ax)

    ax.set_title(f"{player} | Cross Target Zones", fontsize=15, weight="bold")

    return fig


def plot_progressive_actions(player_events, player, team_color="#2563EB"):
    df = player_events[player_events["event_type"].isin(["Pass", "Carry"])].copy()

    if len(df) == 0:
        return None

    df = ensure_xy(df)

    df["end_x"] = np.nan
    df["end_y"] = np.nan

    if "pass_end_location" in df.columns:
        pass_end = df["pass_end_location"].apply(parse_location)
        df["pass_end_x"] = pass_end.apply(lambda v: v[0] if isinstance(v, list) and len(v) > 0 else np.nan)
        df["pass_end_y"] = pass_end.apply(lambda v: v[1] if isinstance(v, list) and len(v) > 1 else np.nan)
        df["end_x"] = df["end_x"].fillna(df["pass_end_x"])
        df["end_y"] = df["end_y"].fillna(df["pass_end_y"])

    if "carry_end_location" in df.columns:
        carry_end = df["carry_end_location"].apply(parse_location)
        df["carry_end_x"] = carry_end.apply(lambda v: v[0] if isinstance(v, list) and len(v) > 0 else np.nan)
        df["carry_end_y"] = carry_end.apply(lambda v: v[1] if isinstance(v, list) and len(v) > 1 else np.nan)
        df["end_x"] = df["end_x"].fillna(df["carry_end_x"])
        df["end_y"] = df["end_y"].fillna(df["carry_end_y"])

    df = df.dropna(subset=["x", "y", "end_x", "end_y"])
    df["progression"] = df["end_x"] - df["x"]
    df = df[df["progression"] >= 15].copy()

    if len(df) == 0:
        return None

    pitch, fig, ax = draw_pitch()

    passes = df[df["event_type"] == "Pass"]
    carries = df[df["event_type"] == "Carry"]

    if len(passes) > 0:
        pitch.arrows(
            passes["x"],
            passes["y"],
            passes["end_x"],
            passes["end_y"],
            ax=ax,
            width=2,
            headwidth=4,
            color=team_color,
            alpha=0.55,
            label="Progressive Pass"
        )

    if len(carries) > 0:
        pitch.arrows(
            carries["x"],
            carries["y"],
            carries["end_x"],
            carries["end_y"],
            ax=ax,
            width=2,
            headwidth=4,
            color="#111827",
            alpha=0.55,
            label="Progressive Carry"
        )

    ax.set_title(f"{player} | Progressive Actions", fontsize=15, weight="bold")
    ax.legend(loc="upper left")

    return fig


def plot_pass_map(player_events, player, team_color="#2563EB"):
    df = player_events[player_events["event_type"] == "Pass"].copy()

    if "pass_end_location" not in df.columns:
        return None

    df = ensure_xy(df)

    pass_end = df["pass_end_location"].apply(parse_location)

    df["end_x"] = pass_end.apply(lambda v: v[0] if isinstance(v, list) and len(v) > 0 else np.nan)
    df["end_y"] = pass_end.apply(lambda v: v[1] if isinstance(v, list) and len(v) > 1 else np.nan)

    df = df.dropna(subset=["x", "y", "end_x", "end_y"])

    if len(df) == 0:
        return None

    pitch, fig, ax = draw_pitch()

    completed = df[df["pass_outcome"].isna()] if "pass_outcome" in df.columns else df
    incomplete = df[df["pass_outcome"].notna()] if "pass_outcome" in df.columns else pd.DataFrame()

    if len(completed) > 0:
        pitch.arrows(
            completed["x"],
            completed["y"],
            completed["end_x"],
            completed["end_y"],
            ax=ax,
            width=2,
            headwidth=4,
            color=team_color,
            alpha=0.55,
            label="Completed"
        )

    if len(incomplete) > 0:
        pitch.arrows(
            incomplete["x"],
            incomplete["y"],
            incomplete["end_x"],
            incomplete["end_y"],
            ax=ax,
            width=2,
            headwidth=4,
            color="#DC2626",
            alpha=0.45,
            label="Incomplete"
        )

    ax.set_title(f"{player} | Pass Map", fontsize=15, weight="bold")
    ax.legend(loc="upper left")

    return fig


def plot_carry_map(player_events, player, team_color="#2563EB"):
    df = player_events[player_events["event_type"] == "Carry"].copy()

    if "carry_end_location" not in df.columns:
        return None

    df = ensure_xy(df)

    carry_end = df["carry_end_location"].apply(parse_location)

    df["end_x"] = carry_end.apply(lambda v: v[0] if isinstance(v, list) and len(v) > 0 else np.nan)
    df["end_y"] = carry_end.apply(lambda v: v[1] if isinstance(v, list) and len(v) > 1 else np.nan)

    df = df.dropna(subset=["x", "y", "end_x", "end_y"])

    if len(df) == 0:
        return None

    pitch, fig, ax = draw_pitch()

    pitch.arrows(
        df["x"],
        df["y"],
        df["end_x"],
        df["end_y"],
        ax=ax,
        width=2,
        headwidth=4,
        color=team_color,
        alpha=0.55
    )

    ax.set_title(f"{player} | Carry Map", fontsize=15, weight="bold")

    return fig


def plot_aerial_duel_zones(player_events, player):
    df = player_events[player_events["event_type"] == "Duel"].copy()

    if "duel_type" in df.columns:
        df = df[df["duel_type"].astype(str).str.contains("Aerial", case=False, na=False)]
    else:
        return None

    return plot_zone_heatmap(
        df,
        ["Duel"],
        f"{player} | Aerial Duel Zones",
        cmap="Greens"
    )


def plot_minutes_timeline(player_events, player, team_color="#2563EB"):
    if len(player_events) == 0 or "minute" not in player_events.columns:
        return None

    df = player_events.copy()
    df["minute_bin"] = (df["minute"] // 15) * 15

    plot_df = (
        df.groupby("minute_bin")
        .size()
        .reset_index(name="events")
        .sort_values("minute_bin")
    )

    fig, ax = plt.subplots(figsize=(8, 4))

    ax.bar(plot_df["minute_bin"].astype(str), plot_df["events"], color=team_color, alpha=0.85)

    ax.set_title(f"{player} | Event Timeline", fontsize=15, weight="bold")
    ax.set_xlabel("Minute Block")
    ax.set_ylabel("Events")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    return fig


# ============================================================
# SIDEBAR
# ============================================================

st.sidebar.title("Filters")

tournament_id = st.sidebar.selectbox("Tournament", get_tournament_options(matches))
team = st.sidebar.selectbox("Team", get_team_options(events, tournament_id))
player = st.sidebar.selectbox("Player", get_player_options(events, tournament_id, team))

player_match_ids = events[
    (events["tournament_id"] == tournament_id) &
    (events["team"] == team) &
    (events["player"] == player)
]["match_id"].dropna().unique()

player_matches = matches[matches["match_id"].isin(player_match_ids)].copy()

if "match_label" not in player_matches.columns:
    player_matches["match_label"] = build_match_label(player_matches)

match_label = st.sidebar.selectbox("Match", player_matches["match_label"].tolist())

match_row = player_matches[player_matches["match_label"] == match_label].iloc[0]
match_id = match_row["match_id"]

opponent = match_row["away_team"] if team == match_row["home_team"] else match_row["home_team"]

match_events = events[events["match_id"] == match_id].copy()
match_events = remove_penalty_shootout(match_events)

player_events = match_events[
    (match_events["team"] == team) &
    (match_events["player"] == player)
].copy()

flag_path, primary_color = get_team_asset(team)

kpis = build_kpis(player_events)


# ============================================================
# HEADER
# ============================================================

col_flag, col_title = st.columns([1, 6])

with col_flag:
    if flag_path:
        st.image(flag_path, width=85)

with col_title:
    st.markdown(
        f"<h1 style='color:{primary_color}; margin-bottom:0;'>{player}</h1>",
        unsafe_allow_html=True
    )
    st.markdown(
        f"<div class='page-subtitle'>{team} vs {opponent} | {match_label} | Player Match Analysis</div>",
        unsafe_allow_html=True
    )


# ============================================================
# KPI ROW 1
# ============================================================

k1, k2, k3, k4, k5, k6 = st.columns(6)

with k1:
    metric_card("Events", kpis["events"])

with k2:
    metric_card("Shots", kpis["shots"])

with k3:
    metric_card("xG", round(kpis["xg"], 2))

with k4:
    metric_card("Passes", kpis["passes"])

with k5:
    metric_card("Carries", kpis["carries"])

with k6:
    metric_card("Pressures", kpis["pressures"])


# ============================================================
# KPI ROW 2
# ============================================================

e1, e2, e3, e4, e5, e6 = st.columns(6)

with e1:
    metric_card("Goals", kpis["goals"])

with e2:
    metric_card("Shot Acc. %", f"{kpis['shot_accuracy_pct']}%")

with e3:
    metric_card("Pass Acc. %", f"{kpis['pass_accuracy_pct']}%")

with e4:
    metric_card("Duel Win %", f"{kpis['duel_win_pct']}%")

with e5:
    metric_card("Crosses", kpis["crosses"])

with e6:
    metric_card("Recoveries", kpis["recoveries"])


# ============================================================
# TABS
# ============================================================

tab_overview, tab_attack, tab_build, tab_defense, tab_stats = st.tabs([
    "Overview",
    "Attack",
    "Build-Up",
    "Defense",
    "Stats & Events"
])


# ============================================================
# OVERVIEW
# ============================================================

with tab_overview:

    st.markdown("## Player Match Summary")

    summary = (
        f"{player} recorded {kpis['events']} total events for {team} against {opponent}. "
        f"He produced {kpis['shots']} shots, {round(kpis['xg'], 2)} xG, "
        f"{kpis['passes']} passes, {kpis['carries']} carries, "
        f"{kpis['pressures']} pressures, {kpis['recoveries']} recoveries/interceptions "
        f"and {kpis['duels']} duels."
    )

    text_card("Auto Match Summary", summary)

    o1, o2 = st.columns(2)

    with o1:
        fig = plot_action_map(player_events, player, team_color=primary_color)

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No action map available.")

    with o2:
        fig = plot_minutes_timeline(player_events, player, team_color=primary_color)

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No event timeline available.")


# ============================================================
# ATTACK
# ============================================================

with tab_attack:

    st.markdown("## Attack")

    a1, a2, a3, a4, a5 = st.columns(5)

    with a1:
        metric_card("Shots", kpis["shots"])

    with a2:
        metric_card("Goals", kpis["goals"])

    with a3:
        metric_card("xG", round(kpis["xg"], 2))

    with a4:
        metric_card("Shot Acc. %", f"{kpis['shot_accuracy_pct']}%")

    with a5:
        metric_card("Cross Acc. %", f"{kpis['cross_accuracy_pct']}%")

    s1, s2 = st.columns(2)

    with s1:
        fig = plot_shot_map(player_events, player, team_color=primary_color)

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No player shots available.")

    with s2:
        fig = plot_zone_heatmap(
            player_events,
            ["Shot"],
            f"{player} | Shot Zones",
            cmap="Reds"
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No shot zones available.")

    st.markdown("### Crosses & Offensive Duels")

    c1, c2 = st.columns(2)

    with c1:
        fig = plot_cross_origin_zones(player_events, player)

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No cross origin zones available.")

    with c2:
        offensive_duels = ensure_xy(player_events)
        offensive_duels = offensive_duels[offensive_duels["x"].fillna(0) >= 60].copy()

        fig = plot_zone_heatmap(
            offensive_duels,
            ["Duel"],
            f"{player} | Offensive Duel Zones",
            cmap="Oranges"
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No offensive duel zones available.")


# ============================================================
# BUILD-UP
# ============================================================

with tab_build:

    st.markdown("## Build-Up & Progression")

    b1, b2, b3, b4 = st.columns(4)

    with b1:
        metric_card("Passes", kpis["passes"])

    with b2:
        metric_card("Pass Acc. %", f"{kpis['pass_accuracy_pct']}%")

    with b3:
        metric_card("Carries", kpis["carries"])

    with b4:
        metric_card("Pass / Carry", round(kpis["passes"] / kpis["carries"], 2) if kpis["carries"] > 0 else 0)

    p1, p2 = st.columns(2)

    with p1:
        fig = plot_pass_map(player_events, player, team_color=primary_color)

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No pass map available.")

    with p2:
        fig = plot_carry_map(player_events, player, team_color=primary_color)

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No carry map available.")

    st.markdown("### Progressive Actions")

    fig = plot_progressive_actions(player_events, player, team_color=primary_color)

    if fig is not None:
        st.pyplot(fig, use_container_width=True)
    else:
        st.info("No progressive actions available.")


# ============================================================
# DEFENSE
# ============================================================

with tab_defense:

    st.markdown("## Defense")

    d1, d2, d3, d4 = st.columns(4)

    with d1:
        metric_card("Pressures", kpis["pressures"])

    with d2:
        metric_card("Recoveries + Interceptions", kpis["recoveries"])

    with d3:
        metric_card("Duels", kpis["duels"])

    with d4:
        metric_card("Fouls", kpis["fouls"])

    r1, r2 = st.columns(2)

    with r1:
        fig = plot_zone_heatmap(
            player_events,
            ["Pressure"],
            f"{player} | Pressure Zones",
            cmap="Oranges"
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No pressure zones available.")

    with r2:
        fig = plot_zone_heatmap(
            player_events,
            ["Ball Recovery", "Interception"],
            f"{player} | Recovery & Interception Zones",
            cmap="Greens"
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No recovery zones available.")

    dcol1, dcol2 = st.columns(2)

    with dcol1:
        fig = plot_zone_heatmap(
            player_events,
            ["Duel"],
            f"{player} | Duel Zones",
            cmap="Purples"
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No duel zones available.")

    with dcol2:
        fig = plot_zone_heatmap(
            player_events,
            ["Foul Committed"],
            f"{player} | Foul Zones",
            cmap="Greys"
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No foul zones available.")


# ============================================================
# STATS & EVENTS
# ============================================================

with tab_stats:

    st.markdown("## Player Match Stats")

    if player_match_stats is not None:
        stats = player_match_stats[
            (player_match_stats["match_id"] == match_id) &
            (player_match_stats["team"] == team) &
            (player_match_stats["player"] == player)
        ].copy()

        if len(stats) > 0:
            st.dataframe(
                stats,
                use_container_width=True,
                hide_index=True
            )
        else:
            st.info("No player match stats available for this match.")
    else:
        st.info("player_match_stats_with_positions not loaded yet.")

    st.markdown("## Compact Event Log")

    event_cols = [
        "minute",
        "second",
        "period",
        "type",
        "event_type",
        "player",
        "team",
        "location",
        "shot_statsbomb_xg",
        "shot_outcome",
        "pass_outcome",
        "pass_length",
        "duel_outcome"
    ]

    existing_cols = [c for c in event_cols if c in player_events.columns]

    st.dataframe(
        player_events[existing_cols],
        use_container_width=True,
        hide_index=True,
        height=520
    )

# ============================================================

# FOOTER

# ============================================================
st.divider()
st.caption(

    "Developed by Juan Ignacio Breccia | Football Analytics · Scouting · Data Science | Powered by StatsBomb Open Data"

)
'''

with open(pages_path / "5_Player_Match.py", "w", encoding="utf-8") as f:
    f.write(player_match_code.strip())

print("5_Player_Match.py updated successfully")

5_Player_Match.py updated successfully


In [439]:
from pathlib import Path

pages_path = Path("streamlit_app/pages")
pages_path.mkdir(parents=True, exist_ok=True)

player_comparison_code = r'''
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import streamlit as st

from utils.data_loader import load_all_data
from utils.filters import (
    get_tournament_options,
    get_team_options,
    get_player_options,
    filter_player
)
from utils.style import apply_global_style
from utils.cards import metric_card, text_card


st.set_page_config(page_title="Player Comparison", layout="wide")
apply_global_style()

data = load_all_data()

matches = data["matches"]
player_master = data["player_master"]
player_similarity = data["player_similarity"]
player_role_fit = data["player_role_fit"]
assets = data["assets"]

BASE_DIR = Path(__file__).resolve().parents[2]


# ============================================================
# HELPERS
# ============================================================

def get_team_asset(team_name):
    asset_row = assets[assets["team"] == team_name]

    if len(asset_row) == 0:
        return None, "#2563EB"

    primary_color = asset_row.iloc[0].get("primary_color", "#2563EB")
    raw_flag_path = asset_row.iloc[0].get("flag_path", None)

    if not raw_flag_path or pd.isna(raw_flag_path):
        return None, primary_color

    raw_flag_path = str(raw_flag_path)

    possible_paths = [
        BASE_DIR / raw_flag_path,
        BASE_DIR / "streamlit_app" / raw_flag_path,
        Path(raw_flag_path),
    ]

    for p in possible_paths:
        if p.exists():
            return str(p), primary_color

    return None, primary_color


def get_row(profile):
    if len(profile) == 0:
        return None
    return profile.iloc[0]


def safe_value(row, col, default=0):
    if row is None:
        return default

    value = row.get(col, default)

    if pd.isna(value):
        return default

    return value


def get_first_available(row, cols, default=0):
    if row is None:
        return default

    for col in cols:
        if col in row.index and pd.notna(row.get(col)):
            return row.get(col)

    return default


def build_player_scores(row):
    if row is None:
        return pd.DataFrame()

    attack = get_first_available(row, ["attacking_volume", "attack_score", "xg", "shots"], 0)
    passing = get_first_available(row, ["passing_volume", "passes", "build_up_score"], 0)
    carrying = get_first_available(row, ["carrying_volume", "carries", "progressive_actions"], 0)
    defense = get_first_available(row, ["defensive_volume", "defense_score", "recoveries", "duels"], 0)
    pressing = get_first_available(row, ["pressures", "pressure_volume", "defensive_activity"], 0)
    importance = get_first_available(row, ["importance_score"], 0)

    df = pd.DataFrame({
        "dimension": ["Attack", "Passing", "Carrying", "Defense", "Pressing", "Importance"],
        "value": [attack, passing, carrying, defense, pressing, importance]
    })

    df["value"] = pd.to_numeric(df["value"], errors="coerce").fillna(0)

    max_value = df["value"].max()

    if max_value > 100:
        df["score"] = (df["value"] / max_value) * 100
    else:
        df["score"] = df["value"]

    df["score"] = df["score"].clip(0, 100)

    return df


def build_comparison_table(row_a, row_b, player_a, player_b):
    metrics = [
        ("Position", "position"),
        ("Role", "general_role"),
        ("Matches", "matches"),
        ("Minutes", "minutes"),
        ("Importance", "importance_score"),
        ("Goals", "goals"),
        ("Assists", "assists"),
        ("xG", "xg"),
        ("xA", "xa"),
        ("Shots", "shots"),
        ("Passes", "passes"),
        ("Carries", "carries"),
        ("Pressures", "pressures"),
        ("Recoveries", "recoveries"),
        ("Duels", "duels"),
        ("Attacking Volume", "attacking_volume"),
        ("Passing Volume", "passing_volume"),
        ("Defensive Volume", "defensive_volume")
    ]

    rows = []

    for label, col in metrics:
        rows.append({
            "Metric": label,
            player_a: safe_value(row_a, col, None),
            player_b: safe_value(row_b, col, None)
        })

    return pd.DataFrame(rows)


def plot_comparison_radar(row_a, row_b, player_a, player_b, color_a="#2563EB", color_b="#DC2626"):
    scores_a = build_player_scores(row_a)
    scores_b = build_player_scores(row_b)

    if len(scores_a) == 0 or len(scores_b) == 0:
        return None

    labels = scores_a["dimension"].tolist()

    values_a = scores_a["score"].astype(float).tolist()
    values_b = scores_b["score"].astype(float).tolist()

    values_a += values_a[:1]
    values_b += values_b[:1]

    angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

    ax.plot(angles, values_a, linewidth=2.5, color=color_a, label=player_a)
    ax.fill(angles, values_a, color=color_a, alpha=0.12)

    ax.plot(angles, values_b, linewidth=2.5, color=color_b, label=player_b)
    ax.fill(angles, values_b, color=color_b, alpha=0.12)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=10, weight="bold")
    ax.set_ylim(0, 100)

    ax.set_title("Player Profile Radar", fontsize=16, weight="bold", pad=20)
    ax.legend(loc="upper right", bbox_to_anchor=(1.25, 1.10))

    return fig


def plot_metric_comparison(row_a, row_b, player_a, player_b, color_a="#2563EB", color_b="#DC2626"):
    metrics = {
        "xG": "xg",
        "Shots": "shots",
        "Passes": "passes",
        "Carries": "carries",
        "Pressures": "pressures",
        "Recoveries": "recoveries",
        "Duels": "duels",
        "Importance": "importance_score"
    }

    rows = []

    for label, col in metrics.items():
        rows.append({
            "metric": label,
            player_a: pd.to_numeric(safe_value(row_a, col, 0), errors="coerce"),
            player_b: pd.to_numeric(safe_value(row_b, col, 0), errors="coerce")
        })

    df = pd.DataFrame(rows).fillna(0)

    fig, ax = plt.subplots(figsize=(10, 4.5))

    x = np.arange(len(df))
    width = 0.36

    ax.bar(x - width / 2, df[player_a], width, label=player_a, color=color_a, alpha=0.85)
    ax.bar(x + width / 2, df[player_b], width, label=player_b, color=color_b, alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(df["metric"])
    ax.set_title("Core Metric Comparison", fontsize=15, weight="bold")
    ax.set_ylabel("Value")
    ax.legend(loc="upper left")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    return fig


def plot_role_fit_bar(rf, player, color="#2563EB"):
    if len(rf) == 0:
        return None

    df = rf.copy()

    role_col = next((c for c in ["role", "role_name", "target_role"] if c in df.columns), None)
    score_col = next((c for c in ["role_fit_score", "fit_score", "score"] if c in df.columns), None)

    if role_col is None or score_col is None:
        return None

    df[score_col] = pd.to_numeric(df[score_col], errors="coerce").fillna(0)
    df = df.sort_values(score_col, ascending=False).head(6)

    fig, ax = plt.subplots(figsize=(8, 4))

    ax.barh(df[role_col][::-1], df[score_col][::-1], color=color, alpha=0.85)

    ax.set_title(f"{player} | Role Fit", fontsize=15, weight="bold")
    ax.set_xlabel("Score")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    return fig


def plot_similarity_bar(sim, player, color="#2563EB"):
    if len(sim) == 0:
        return None

    df = sim.copy()

    player_col = next((c for c in ["similar_player", "player_2", "similar_name"] if c in df.columns), None)
    score_col = next((c for c in ["similarity_score", "score"] if c in df.columns), None)

    if player_col is None:
        return None

    if score_col is None:
        if "similarity_rank" in df.columns:
            df["similarity_score_plot"] = 100 - pd.to_numeric(df["similarity_rank"], errors="coerce").fillna(100)
            score_col = "similarity_score_plot"
        else:
            return None

    df[score_col] = pd.to_numeric(df[score_col], errors="coerce").fillna(0)

    if "similarity_rank" in df.columns:
        df = df.sort_values("similarity_rank")
    else:
        df = df.sort_values(score_col, ascending=False)

    df = df.head(6)

    fig, ax = plt.subplots(figsize=(8, 4))

    ax.barh(df[player_col][::-1], df[score_col][::-1], color=color, alpha=0.85)

    ax.set_title(f"{player} | Similar Players", fontsize=15, weight="bold")
    ax.set_xlabel("Similarity")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    return fig


def render_player_card(player, team, row, color):
    if row is None:
        st.info(f"No profile available for {player}.")
        return

    text_card(
        f"{player} Profile",
        (
            f"Team: {team} | "
            f"Position: {safe_value(row, 'position', 'N/A')} | "
            f"Role: {safe_value(row, 'general_role', 'N/A')} | "
            f"Importance: {round(safe_value(row, 'importance_score', 0), 2)} | "
            f"xG: {round(safe_value(row, 'xg', 0), 2)}"
        )
    )


# ============================================================
# SIDEBAR
# ============================================================

st.sidebar.title("Filters")

tournaments = get_tournament_options(matches)

# ------------------------------------------------
# PLAYER A
# ------------------------------------------------

tournament_a = st.sidebar.selectbox(
    "Tournament A",
    tournaments,
    index=0
)

teams_a = get_team_options(
    player_master,
    tournament_a
)

team_a = st.sidebar.selectbox(
    "Team A",
    teams_a,
    index=0
)

players_a = get_player_options(
    player_master,
    tournament_a,
    team_a
)

player_a = st.sidebar.selectbox(
    "Player A",
    players_a,
    index=0
)

# ------------------------------------------------
# PLAYER B
# ------------------------------------------------

tournament_b = st.sidebar.selectbox(
    "Tournament B",
    tournaments,
    index=1 if len(tournaments) > 1 else 0
)

teams_b = get_team_options(
    player_master,
    tournament_b
)

team_b = st.sidebar.selectbox(
    "Team B",
    teams_b,
    index=0
)

players_b = get_player_options(
    player_master,
    tournament_b,
    team_b
)

player_b = st.sidebar.selectbox(
    "Player B",
    players_b,
    index=0
)

profile_a = filter_player(
    player_master,
    tournament_a,
    team_a,
    player_a
)

profile_b = filter_player(
    player_master,
    tournament_b,
    team_b,
    player_b
)

row_a = get_row(profile_a)
row_b = get_row(profile_b)

flag_a, color_a = get_team_asset(team_a)
flag_b, color_b = get_team_asset(team_b)


# ============================================================
# FILTERED EXTRA DATA
# ============================================================

rf_a = player_role_fit[
    (player_role_fit["tournament_id"] == tournament_a)
    &
    (player_role_fit["team"] == team_a)
    &
    (player_role_fit["player"] == player_a)
].copy()

rf_b = player_role_fit[
    (player_role_fit["tournament_id"] == tournament_b)
    &
    (player_role_fit["team"] == team_b)
    &
    (player_role_fit["player"] == player_b)
].copy()

sim_a = player_similarity[
    (player_similarity["tournament_id"] == tournament_a)
    &
    (player_similarity["team"] == team_a)
    &
    (player_similarity["player"] == player_a)
].copy()

sim_b = player_similarity[
    (player_similarity["tournament_id"] == tournament_b)
    &
    (player_similarity["team"] == team_b)
    &
    (player_similarity["player"] == player_b)
].copy()

# ============================================================
# HEADER
# ============================================================

h1, h2, h3 = st.columns([1, 6, 1])

with h1:
    if flag_a:
        st.image(flag_a, width=85)

with h2:
    st.markdown(
        f"<h1 style='text-align:center; margin-bottom:0;'>{player_a} vs {player_b}</h1>",
        unsafe_allow_html=True
    )

    st.markdown(
        f"""
        <div class='page-subtitle' style='text-align:center;'>
        {team_a} ({tournament_a})
        vs
        {team_b} ({tournament_b})
        </div>
        """,
        unsafe_allow_html=True
    )

with h3:
    if flag_b:
        st.image(flag_b, width=85)

# ============================================================
# KPI ROW
# ============================================================

k1, k2, k3, k4, k5, k6 = st.columns(6)

with k1:
    metric_card(f"{player_a} Role", safe_value(row_a, "general_role", "N/A"))

with k2:
    metric_card(f"{player_b} Role", safe_value(row_b, "general_role", "N/A"))

with k3:
    metric_card(f"{player_a} Importance", round(safe_value(row_a, "importance_score", 0), 2))

with k4:
    metric_card(f"{player_b} Importance", round(safe_value(row_b, "importance_score", 0), 2))

with k5:
    metric_card(f"{player_a} xG", round(safe_value(row_a, "xg", 0), 2))

with k6:
    metric_card(f"{player_b} xG", round(safe_value(row_b, "xg", 0), 2))


# ============================================================
# SINGLE MAIN VIEW
# ============================================================

left_col, radar_col, right_col = st.columns([1.05, 1.35, 1.05])

with left_col:
    render_player_card(player_a, team_a, row_a, color_a)

with radar_col:
    fig = plot_comparison_radar(
        row_a,
        row_b,
        player_a,
        player_b,
        color_a=color_a,
        color_b=color_b
    )

    if fig is not None:
        st.pyplot(fig, use_container_width=True)

with right_col:
    render_player_card(player_b, team_b, row_b, color_b)


st.markdown("### Core Metric Comparison")

fig = plot_metric_comparison(
    row_a,
    row_b,
    player_a,
    player_b,
    color_a=color_a,
    color_b=color_b
)

if fig is not None:
    st.pyplot(fig, use_container_width=True)


# ============================================================
# DETAILS
# ============================================================

tab_metrics, tab_roles, tab_similarity = st.tabs([
    "Metrics",
    "Role Fit",
    "Similarity"
])


with tab_metrics:

    comparison_table = build_comparison_table(row_a, row_b, player_a, player_b)

    st.dataframe(
        comparison_table,
        use_container_width=True,
        hide_index=True
    )


with tab_roles:

    c1, c2 = st.columns(2)

    with c1:

        st.markdown(
            f"### {player_a} | {tournament_a}"
        )

        fig = plot_role_fit_bar(
            rf_a,
            player_a,
            color=color_a
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)

    with c2:

        st.markdown(
            f"### {player_b} | {tournament_b}"
        )

        fig = plot_role_fit_bar(
            rf_b,
            player_b,
            color=color_b
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)


with tab_similarity:

    c1, c2 = st.columns(2)

    with c1:

        st.markdown(
            f"### {player_a} | {tournament_a}"
        )

        fig = plot_similarity_bar(
            sim_a,
            player_a,
            color=color_a
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)

    with c2:

        st.markdown(
            f"### {player_b} | {tournament_b}"
        )

        fig = plot_similarity_bar(
            sim_b,
            player_b,
            color=color_b
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
# ============================================================

# FOOTER

# ============================================================
st.divider()
st.caption(

    "Developed by Juan Ignacio Breccia | Football Analytics · Scouting · Data Science | Powered by StatsBomb Open Data"

)
'''

with open(pages_path / "6_Player_Comparison.py", "w", encoding="utf-8") as f:
    f.write(player_comparison_code.strip())

print("6_Player_Comparison.py updated successfully")

6_Player_Comparison.py updated successfully


In [371]:
from pathlib import Path

visuals_path = Path("streamlit_app/utils/visuals.py")

text = visuals_path.read_text(encoding="utf-8")

# Busca líneas con \n literal
for i, line in enumerate(text.splitlines(), start=1):
    if "\\n" in line:
        print(i, repr(line))

In [373]:
import py_compile

py_compile.compile(
    "streamlit_app/utils/visuals.py",
    doraise=True
)

print("visuals.py OK")

visuals.py OK


In [375]:
from pathlib import Path

visuals_path = Path("streamlit_app/utils/visuals.py")

lines = visuals_path.read_text(encoding="utf-8").splitlines()

clean_lines = [
    line for line in lines
    if line.strip() not in ["\\n", "\\n\\n"]
]

visuals_path.write_text("\n".join(clean_lines) + "\n", encoding="utf-8")

print("visuals.py limpiado")

visuals.py limpiado


In [377]:
import py_compile

py_compile.compile(
    "streamlit_app/utils/visuals.py",
    doraise=True
)

print("visuals.py OK")

visuals.py OK
